# AgriWise — Complete Modeling & Proxy Pipeline V3.3

**Scope:** CALABARZON · Rice/Palay · Tomato · Red Onion · Banana

This notebook supersedes V3.2 as the recommended **single master modeling notebook**.

It is designed around the datasets that are actually available:

- PSA crop production and area planted/harvested
- PSA monthly farmgate prices
- FIES-LFS 2023 household summary and dictionary
- Monthly LFS PUF files
- optional PSA commodity-flow API

## V3.3 modeling policy

The notebook distinguishes four different things:

1. **Supply forecast** — physical production in metric tons.
2. **Farmgate-price forecast** — source price units, normally ₱/kg where applicable.
3. **Household demand proxy** — FIES expenditure-category estimation, not physical commodity demand.
4. **Quarterly demand-pressure proxy** — a derived index using FIES baseline + socioeconomic/affordability indicators. It is an *estimated proxy*, not observed quarterly consumption.

No component is allowed to manufacture observations merely to obtain a forecast.

### FIES target mapping used

| AgriWise commodity | FIES target | Meaning in AgriWise |
|---|---|---|
| Rice | `BREAD` | Cereal/bread expenditure proxy; not rice-only |
| Tomato | `VEG` | Broad vegetable expenditure proxy |
| Red Onion | `VEG` | Broad vegetable expenditure proxy |
| Banana | `FRUIT` | Broad fruit expenditure proxy |

Tomato and Red Onion therefore share the same broad FIES target unless a more granular consumption source becomes available.

### Important V3.3 changes

- **Supply leakage fix:** same-period `area` and target-derived `yield_proxy` are no longer model predictors. Only lagged agronomic values are allowed.
- **Independent final holdout:** supply/price candidate selection is performed using recent rolling-origin validation on the pre-holdout training period; the most recent holdout is used only for final reporting.
- **Recent stability validation:** rolling validation is dense over recent periods instead of six widely separated single-period checkpoints across the full history.
- **Province-aware time-series models:** `geolocation` is one-hot encoded inside the model pipeline.
- **Robust/nonnegative candidates:** Huber Gradient Boosting and Poisson HistGradientBoosting are included alongside HistGB, Random Forest, and seasonal naive.
- **Demand selection fix:** FIES proxy candidates are selected using stratified cross-validation inside the training partition, then evaluated once on an untouched stratified holdout.
- **Demand-pressure independence:** local supply seasonality is exported as context but is no longer multiplied into the demand-pressure formula; a zero local harvest therefore cannot force estimated demand pressure to zero.
- `0` farmgate prices remain treated as **missing observations**, not real ₱0 prices.
- FIES identifiers and target-derived expenditure totals remain excluded from demand predictors; `RFACT` remains a survey weight rather than a predictor.

> **Outputs are intentionally cleared in this V3.3 file. Run All before interpreting or deploying any metrics/artifacts.**


In [ ]:

# Optional installation if your environment is missing packages:
# %pip install -q pandas numpy scikit-learn xgboost joblib openpyxl requests matplotlib

from pathlib import Path
from datetime import datetime
import json, re, zipfile, shutil, warnings, math

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

COMMODITIES = ["rice", "tomato", "red_onion", "banana"]
CALABARZON = ["Batangas", "Cavite", "Laguna", "Quezon", "Rizal"]

MONTHS = {
    "January":1, "February":2, "March":3, "April":4,
    "May":5, "June":6, "July":7, "August":8,
    "September":9, "October":10, "November":11, "December":12,
}

# FIES RPROV codes from the supplied 2023 valueset.
FIES_PROVINCE_MAP = {
    1000: "Batangas",
    2100: "Cavite",
    3400: "Laguna",
    5600: "Quezon",
    5624: "Quezon",   # Lucena City is grouped with Quezon for AgriWise province-level mapping.
    5800: "Rizal",
}

DEMAND_TARGETS = {
    "rice": "BREAD",
    "tomato": "VEG",
    "red_onion": "VEG",
    "banana": "FRUIT",
}

# Keep these explicit. They are economic/demographic predictors, not target-derived food totals.
DEMAND_NUMERIC_FEATURES = [
    "FSIZE", "URB", "TOINC", "PERCAPITA", "EAINC", "WAGES",
    "REG_SAL", "SEASON_SAL", "NET_CFG", "NETSHARE",
    "CASH_ABROAD", "CASH_DOMESTIC", "PENSION", "RENTALS_REC",
    "INTEREST", "NET_LPR", "NET_FISH", "NET_FOR", "NET_RET",
    "NET_MFG", "NET_TRANS",
]
DEMAND_CATEGORICAL_FEATURES = ["RPROV"]

# --- V3.3 additions: weighted demand training, an XGBoost benchmark race, and
# EF_agriwise forecast-service artifact-contract compatibility. ---

# Matches EF_agriwise/ml/demand/pipeline/train_quarterly_101.py exactly, so the
# GradientBoosting-vs-XGBoost demand race is an apples-to-apples comparison
# against the team's established FIES/XGBoost reference pattern (ross'_work).
DEMAND_XGB_PARAMETERS = {
    "n_estimators": 900,
    "learning_rate": 0.01,
    "max_depth": 2,
    "min_child_weight": 12,
    "subsample": 0.65,
    "colsample_bytree": 0.50,
    "gamma": 0.05,
    "reg_alpha": 0.50,
    "reg_lambda": 5.0,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Must match EF_agriwise/ml/forecasting/artifact_registry.py::slugify_commodity
# exactly -- this is the filename convention the forecast-service registry expects.
def slugify_commodity(commodity):
    return commodity.strip().lower().replace(" ", "_")

# Must match EF_agriwise/ml/forecasting/domain.py::COMMODITIES exactly (Title Case).
COMMODITY_TITLE_CASE = {
    "rice": "Rice",
    "tomato": "Tomato",
    "red_onion": "Red Onion",
    "banana": "Banana",
}

# The FIES VEG demand model is shared across Tomato and Red Onion, so the one
# trained artifact is promoted into two separate commodity-keyed bundles.
DEMAND_ARTIFACT_COMMODITIES = {
    "rice": ["Rice"],
    "vegetable_shared": ["Tomato", "Red Onion"],
    "banana": ["Banana"],
}

def find_ef_agriwise_root():
    here = Path.cwd()
    for candidate in [here / "EF_agriwise", here.parent / "EF_agriwise", here.parent.parent / "EF_agriwise"]:
        if (candidate / "ml" / "forecasting" / "artifact_registry.py").exists():
            return candidate.resolve()
    return None

# Where trained artifacts are promoted so EF_agriwise/ml/forecasting/artifact_registry.py
# can load them directly. Resolved relative to this notebook's working directory;
# promotion is skipped gracefully (see the final cell) if EF_agriwise isn't found.
EF_AGRIWISE_ROOT = find_ef_agriwise_root()
EF_ARTIFACTS_ROOT = (EF_AGRIWISE_ROOT / "ml" / "artifacts") if EF_AGRIWISE_ROOT else None
if EF_ARTIFACTS_ROOT is None:
    print("EF_agriwise not found relative to this notebook; artifact promotion will be skipped at the end of the run.")
else:
    print("EF_agriwise artifacts will be promoted to:", EF_ARTIFACTS_ROOT)

# V3.4: promotion into the live EF_agriwise service is gated behind this flag so a
# routine "Run All" during iterative tuning never overwrites production artifacts.
# Flip to True only after reviewing this run's deployment-summary verdicts.
PROMOTE_TO_EF_AGRIWISE = False

ARTIFACT_ROOT = Path("artifacts_v3_4")
for d in [
    ARTIFACT_ROOT / "supply",
    ARTIFACT_ROOT / "price",
    ARTIFACT_ROOT / "demand",
    ARTIFACT_ROOT / "prepared",
    ARTIFACT_ROOT / "reports",
    ARTIFACT_ROOT / "config",
]:
    d.mkdir(parents=True, exist_ok=True)

print("Artifact root:", ARTIFACT_ROOT.resolve())


## 1. Locate/extract the supplied datasets

In [2]:

# The notebook supports either extracted folders or ZIP files.
# Put it inside your AgriDataSet folder, repository root, or point BASE_DIR manually.

BASE_DIR = Path.cwd()

def find_one(patterns, roots=None):
    roots = roots or [BASE_DIR, BASE_DIR / "AgriDataSet", BASE_DIR / "data", BASE_DIR / "ml"]
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            hits = list(root.rglob(pat))
            if hits:
                return hits[0]
    return None

def extract_zip_if_needed(zip_path, label):
    if zip_path is None:
        return None
    dest = BASE_DIR / ".agriwise_cache" / label
    marker = dest / ".complete"
    if not marker.exists():
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
        marker.write_text("ok", encoding="utf-8")
    return dest

crop_dir = find_one(["psa_openstat_crops"])
price_dir = find_one(["psa_price_downloads"])
lfs_root = find_one(["lfsFiles"])

if crop_dir is None or not crop_dir.is_dir():
    crop_zip = find_one(["psa_openstat_crops*.zip"])
    cache = extract_zip_if_needed(crop_zip, "crops")
    crop_dir = find_one(["psa_openstat_crops"], [cache]) if cache else None

if price_dir is None or not price_dir.is_dir():
    price_zip = find_one(["psa_price_downloads*.zip"])
    cache = extract_zip_if_needed(price_zip, "prices")
    price_dir = find_one(["psa_price_downloads"], [cache]) if cache else None

if lfs_root is None or not lfs_root.is_dir():
    lfs_zip = find_one(["lfsFiles*.zip"])
    cache = extract_zip_if_needed(lfs_zip, "lfs")
    lfs_root = find_one(["lfsFiles"], [cache]) if cache else None

print("crop_dir :", crop_dir)
print("price_dir:", price_dir)
print("lfs_root :", lfs_root)

assert crop_dir is not None, "PSA crop dataset folder/ZIP not found."
assert price_dir is not None, "PSA price dataset folder/ZIP not found."
assert lfs_root is not None, "FIES/LFS dataset folder/ZIP not found."


crop_dir : c:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\psa_openstat_crops
price_dir: c:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\psa_price_downloads
lfs_root : c:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\lfsFiles


## 2. Dataset inventory and structural audit

In [3]:

def csv_shape(path):
    try:
        sample = pd.read_csv(path, nrows=5, low_memory=False)
        # Count rows cheaply only when needed later; inventory uses columns + file size.
        return len(sample.columns)
    except Exception:
        return np.nan

inventory = []
for family, root in [("crops", crop_dir), ("prices", price_dir), ("fies_lfs", lfs_root)]:
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".csv", ".xlsx"}:
            inventory.append({
                "family": family,
                "file": p.name,
                "size_mb": round(p.stat().st_size / (1024**2), 2),
                "columns_sampled": csv_shape(p) if p.suffix.lower() == ".csv" else np.nan,
            })

inventory_df = pd.DataFrame(inventory).sort_values(["family", "file"])
display(inventory_df)
inventory_df.to_csv(ARTIFACT_ROOT / "reports" / "dataset_inventory.csv", index=False)


,family,file,size_mb,columns_sampled
0,crops,"Fruit Crops- Area Planted-Harvested, by Region...",24.88,121.0
1,crops,Fruit Crops- Number of Bearing Trees-Hills-Vin...,22.56,121.0
2,crops,"Fruit Crops- Volume of Production, by Region, ...",25.53,121.0
3,crops,Major Crops- Number of Bearing Trees-Hills-Vin...,0.22,121.0
4,crops,Palay and Corn- Area Harvested in Hectares by ...,1.70,282.0
...,...,...,...,...
12,prices,"Cereals- Farmgate Prices by Geolocation, Commo...",3.44,223.0
13,prices,"Condiments- Farmgate Prices by Geolocation, Co...",6.60,223.0
14,prices,Fruit Vegetables- Farmgate Prices by Geolocati...,8.20,223.0
15,prices,"Fruits- Farmgate Prices by Commodity, by Regio...",30.71,223.0


## 3. Shared parsing and validation helpers

In [4]:

def clean_geo(v):
    return re.sub(r"^\.+", "", str(v)).strip()

def numeric_clean(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .replace({"..": np.nan, "...": np.nan, "nan": np.nan, "": np.nan}),
        errors="coerce",
    )

def parse_quarter_column(col):
    m = re.search(r"(\d{4}).*Quarter\s*([1-4])", str(col), re.I)
    if not m:
        return None
    return pd.Timestamp(int(m.group(1)), (int(m.group(2))-1)*3 + 1, 1)

def parse_month_column(col):
    m = re.match(r"^(\d{4})\s+(" + "|".join(MONTHS) + r")$", str(col))
    if not m:
        return None
    return pd.Timestamp(int(m.group(1)), MONTHS[m.group(2)], 1)

def regression_metrics(y, pred, seasonal_naive=None):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = np.isfinite(y) & np.isfinite(pred)
    y, pred = y[mask], pred[mask]
    if len(y) == 0:
        return {}
    err = pred - y
    mae = mean_absolute_error(y, pred)
    mse = mean_squared_error(y, pred)
    rmse = math.sqrt(mse)
    r2 = r2_score(y, pred) if len(y) > 1 and np.nanstd(y) > 1e-12 else np.nan
    nz = np.abs(y) > 1e-12
    mape = np.mean(np.abs(err[nz] / y[nz])) * 100 if nz.any() else np.nan
    denom = np.abs(y) + np.abs(pred)
    sm = denom > 1e-12
    smape = np.mean(2 * np.abs(err[sm]) / denom[sm]) * 100 if sm.any() else np.nan
    wape = np.sum(np.abs(err)) / np.sum(np.abs(y)) * 100 if np.sum(np.abs(y)) > 1e-12 else np.nan
    bias = np.mean(err)
    mase = np.nan
    if seasonal_naive is not None:
        naive = np.asarray(seasonal_naive, dtype=float)[mask]
        naive_mae = np.nanmean(np.abs(y - naive))
        if np.isfinite(naive_mae) and naive_mae > 1e-12:
            mase = mae / naive_mae
    return {
        "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2,
        "MAPE": mape, "sMAPE": smape, "WAPE": wape,
        "MASE": mase, "Bias": bias, "N": len(y),
        "ZeroRatePct": float(np.mean(np.isclose(y, 0))*100),
        "NonZeroN": int(np.sum(~np.isclose(y, 0))),
    }

def weighted_regression_metrics(y, pred, weights):
    """Survey-weighted metrics, matching EF_agriwise/ml/demand/pipeline's
    metrics() function (sample_weight=RFACT) so demand-proxy numbers are
    directly comparable to the team's FIES/XGBoost reference models."""
    y_raw = np.asarray(y, dtype=float)
    pred_raw = np.asarray(pred, dtype=float)
    w_raw = np.asarray(weights, dtype=float)
    nonfinite_predictions = int((~np.isfinite(pred_raw)).sum())
    negative_predictions = int((pred_raw < 0).sum())
    mask = np.isfinite(y_raw) & np.isfinite(pred_raw) & np.isfinite(w_raw) & (w_raw > 0)
    y, pred, w = y_raw[mask], pred_raw[mask], w_raw[mask]
    if len(y) == 0:
        return {}
    actual_mean = float(np.average(y, weights=w))
    mae = float(mean_absolute_error(y, pred, sample_weight=w))
    rmse = float(math.sqrt(mean_squared_error(y, pred, sample_weight=w)))
    r2 = float(r2_score(y, pred, sample_weight=w)) if len(y) > 1 else np.nan
    actual_total = float(np.sum(y * w))
    predicted_total = float(np.sum(pred * w))
    return {
        "r2": r2, "mae": mae, "rmse": rmse,
        "normalized_mae_percent": (mae / actual_mean * 100) if actual_mean else np.nan,
        "aggregate_bias_percent": ((predicted_total - actual_total) / actual_total * 100) if actual_total else np.nan,
        "target_mean": actual_mean, "prediction_mean": float(np.average(pred, weights=w)),
        "n": int(len(y)), "negative_predictions": negative_predictions,
        "nonfinite_predictions": nonfinite_predictions,
    }

def demand_verdict(weighted_r2, wape):
    """Realistic, documented tiers for cross-sectional FIES demand-proxy models --
    replaces the previous unconditional 'USABLE_PROXY' label. Bands are deliberately
    conservative: FIES category proxies are noisy household survey data, not an
    observed commodity time series."""
    if not np.isfinite(weighted_r2) and not np.isfinite(wape):
        return "INSUFFICIENT_DATA", "no valid holdout error"
    if (np.isfinite(weighted_r2) and weighted_r2 >= 0.45) or (np.isfinite(wape) and wape <= 30):
        return "USABLE_PROXY", "weighted R2 and/or WAPE meet the usable-proxy bar"
    if (np.isfinite(weighted_r2) and weighted_r2 >= 0.20) or (np.isfinite(wape) and wape <= 55):
        return "INDICATIVE_PROXY", "weak explanatory power; use for relative/directional signal, not precise point forecasts"
    return "INSUFFICIENT_DATA", "forecast errors too high to support even an indicative proxy"

def add_panel_lags(df, seasonal_period, lags):
    x = df.sort_values(["geolocation", "date"]).copy()
    g = x.groupby("geolocation", group_keys=False)["target"]
    for lag in lags:
        x[f"lag_{lag}"] = g.shift(lag)
    x["roll_mean_3"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    x["roll_std_3"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
    x["season_sin"] = np.sin(2*np.pi*x["season"] / seasonal_period)
    x["season_cos"] = np.cos(2*np.pi*x["season"] / seasonal_period)
    return x


## 4. Build supply and agronomic auxiliary panels

In [5]:

CROP_FILES = {p.name: p for p in crop_dir.rglob("*.csv")}

def wide_quarter_panel(path, commodity_col, commodity_pattern, value_name):
    df = pd.read_csv(path, low_memory=False)
    df["_geo"] = df["Geolocation"].map(clean_geo)
    mask = df[commodity_col].astype(str).str.contains(commodity_pattern, case=False, regex=True, na=False)
    df = df[mask & df["_geo"].isin(CALABARZON)].copy()

    rows = []
    for c in df.columns:
        dt = parse_quarter_column(c)
        if dt is None:
            continue
        vals = numeric_clean(df[c])
        tmp = pd.DataFrame({
            "geolocation": df["_geo"],
            "date": dt,
            value_name: vals,
        })
        rows.append(tmp)

    if not rows:
        return pd.DataFrame()

    long = pd.concat(rows, ignore_index=True)
    # Duplicated wide PSA blocks are consolidated. Missing is not converted to zero.
    long = (
        long.groupby(["geolocation", "date"], as_index=False)[value_name]
            .mean()
            .sort_values(["geolocation", "date"])
    )
    return long

palay_prod = wide_quarter_panel(
    CROP_FILES[[n for n in CROP_FILES if n.startswith("Palay and Corn- Volume")][0]],
    "Ecosystem/Croptype", r"Palay", "production"
)
palay_area = wide_quarter_panel(
    CROP_FILES[[n for n in CROP_FILES if n.startswith("Palay and Corn- Area")][0]],
    "Ecosystem/Croptype", r"Palay", "area"
)
veg_prod_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Vegetables and Root Crops- Volume")][0]]
veg_area_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Vegetables and Root Crops- Area")][0]]
fruit_prod_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Fruit Crops- Volume")][0]]
fruit_area_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Fruit Crops- Area")][0]]

tomato_prod = wide_quarter_panel(veg_prod_file, "Crop", r"^Tomato$", "production")
tomato_area = wide_quarter_panel(veg_area_file, "Crop", r"^Tomato$", "area")
onion_prod = wide_quarter_panel(veg_prod_file, "Crop", r"Onion.*red|Red.*onion|Onion", "production")
onion_area = wide_quarter_panel(veg_area_file, "Crop", r"Onion.*red|Red.*onion|Onion", "area")
banana_prod = wide_quarter_panel(fruit_prod_file, "Crop", r"^Banana$", "production")
banana_area = wide_quarter_panel(fruit_area_file, "Crop", r"^Banana$", "area")

supply_raw = {
    "rice": palay_prod,
    "tomato": tomato_prod,
    "red_onion": onion_prod,
    "banana": banana_prod,
}
area_raw = {
    "rice": palay_area,
    "tomato": tomato_area,
    "red_onion": onion_area,
    "banana": banana_area,
}

supply_panels = {}
for commodity in COMMODITIES:
    p = supply_raw[commodity].rename(columns={"production":"target"})
    a = area_raw[commodity]
    x = p.merge(a, on=["geolocation","date"], how="left")
    x["yield_proxy"] = np.where(x["area"] > 0, x["target"] / x["area"], np.nan)
    x["year"] = x["date"].dt.year
    x["season"] = x["date"].dt.quarter
    supply_panels[commodity] = x

supply_audit = pd.concat([
    pd.DataFrame({
        "commodity":[c],
        "rows":[len(x)],
        "date_min":[x["date"].min()],
        "date_max":[x["date"].max()],
        "missing_target_pct":[x["target"].isna().mean()*100],
        "zero_target_pct":[np.isclose(x["target"].fillna(np.nan),0).mean()*100],
        "locations":[x["geolocation"].nunique()],
    })
    for c,x in supply_panels.items()
], ignore_index=True)
display(supply_audit)
supply_audit.to_csv(ARTIFACT_ROOT / "reports" / "supply_data_audit.csv", index=False)


,commodity,rows,date_min,date_max,missing_target_pct,zero_target_pct,locations
0,rice,800,1987-01-01,2026-10-01,1.250000,0.000000,5
1,tomato,340,2010-01-01,2026-10-01,2.941176,10.588235,5
2,red_onion,340,2010-01-01,2026-10-01,2.941176,45.000000,5
3,banana,340,2010-01-01,2026-10-01,2.941176,0.000000,5


## 5. Build monthly farmgate-price panels — **zero-price fix included**

In [6]:

PRICE_FILES = {p.name: p for p in price_dir.rglob("*.csv")}

PRICE_SPECS = {
    "rice": (
        [n for n in PRICE_FILES if n.startswith("Cereals- Farmgate")][0],
        r"Palay \[Paddy\]",
    ),
    "tomato": (
        [n for n in PRICE_FILES if n.startswith("Fruit Vegetables- Farmgate")][0],
        r"^Tomato$",
    ),
    "red_onion": (
        [n for n in PRICE_FILES if n.startswith("Condiments- Farmgate")][0],
        r"Onion, red",
    ),
    "banana": (
        [n for n in PRICE_FILES if n.startswith("Fruits- Farmgate")][0],
        r"Banana",
    ),
}

def build_price_panel(path, pattern):
    df = pd.read_csv(path, low_memory=False)
    df["_geo"] = df["Geolocation"].map(clean_geo)
    df = df[
        df["Commodity"].astype(str).str.contains(pattern, case=False, regex=True, na=False)
        & df["_geo"].isin(CALABARZON)
    ].copy()

    rows = []
    for c in df.columns:
        dt = parse_month_column(c)
        if dt is None:
            continue
        val = numeric_clean(df[c])
        # Critical fix: a real farmgate price cannot be represented as zero here.
        # PSA tables use explicit zeros in many sparse rows; treat <=0 as unavailable.
        val = val.mask(val <= 0)
        rows.append(pd.DataFrame({
            "geolocation": df["_geo"],
            "date": dt,
            "price": val,
            "variant": df["Commodity"].astype(str),
        }))

    long = pd.concat(rows, ignore_index=True)

    # Multiple varieties can exist. First collapse duplicates within variety/date,
    # then take median across available varieties for a robust commodity-level price.
    long = (
        long.groupby(["geolocation","date","variant"], as_index=False)["price"].mean()
    )
    panel = (
        long.groupby(["geolocation","date"], as_index=False)["price"].median()
            .rename(columns={"price":"target"})
            .sort_values(["geolocation","date"])
    )
    panel["year"] = panel["date"].dt.year
    panel["season"] = panel["date"].dt.month
    return panel

price_panels = {
    c: build_price_panel(PRICE_FILES[fname], patt)
    for c,(fname,patt) in PRICE_SPECS.items()
}

price_audit_rows = []
for c,x in price_panels.items():
    expected = x["geolocation"].nunique() * x["date"].nunique()
    price_audit_rows.append({
        "commodity": c,
        "rows_with_grid_keys": len(x),
        "nonmissing_price_n": int(x["target"].notna().sum()),
        "missing_price_pct": float(x["target"].isna().mean()*100),
        "zero_price_after_cleaning_n": int(np.isclose(x["target"].fillna(np.nan),0).sum()),
        "date_min": x["date"].min(),
        "date_max": x["date"].max(),
        "locations": x["geolocation"].nunique(),
    })

price_audit = pd.DataFrame(price_audit_rows)
display(price_audit)
price_audit.to_csv(ARTIFACT_ROOT / "reports" / "price_data_audit.csv", index=False)

for c,x in price_panels.items():
    x.to_csv(ARTIFACT_ROOT / "prepared" / f"{c}_price_panel.csv", index=False)


,commodity,rows_with_grid_keys,nonmissing_price_n,missing_price_pct,zero_price_after_cleaning_n,date_min,date_max,locations
0,rice,1020,988,3.137255,0,2010-01-01,2026-12-01,5
1,tomato,1020,798,21.764706,0,2010-01-01,2026-12-01,5
2,red_onion,1020,48,95.294118,0,2010-01-01,2026-12-01,5
3,banana,1020,990,2.941176,0,2010-01-01,2026-12-01,5


### Why Red Onion price previously failed

The supplied raw condiments table contains many explicit `0.00` monthly values for red-onion rows.  
Those must **not** be trained as actual farmgate prices. V3.3 converts non-positive prices to missing before modeling.

This will usually improve Red Onion diagnostics substantially, although the series may still be too sparse for province-level forecasting.


## 6. Join supply ↔ price and create **strictly leakage-safe** lag features

V3.3 keeps the current-period production target isolated from its predictors.

For supply forecasting, `area` and `yield_proxy = production / area` are retained in the prepared panel for auditing and for **future lag construction**, but the model may only consume their historical values (`area_lag1`, `area_lag4`, `yield_proxy_lag1`, `yield_proxy_lag4`). Same-period area/yield is deliberately excluded because the same-period yield proxy mathematically contains the target.

Price models continue to use lagged supply/area/yield context only.


In [7]:
def quarterly_to_monthly_lagged(qdf, value_col, out_col):
    q = qdf[["geolocation","date",value_col]].copy()
    q["quarter_end"] = q["date"] + pd.offsets.QuarterEnd(0)
    q = q.rename(columns={value_col:out_col})
    return q[["geolocation","quarter_end",out_col]]

# Supply features: strictly historical agronomic context + lagged quarterly price.
# IMPORTANT: same-period area/yield_proxy are retained in the dataframe for audit/history,
# but feature_columns() below never exposes them to the supply model.
for c in COMMODITIES:
    s = supply_panels[c].sort_values(["geolocation","date"]).copy()
    p = price_panels[c].copy()
    p["qdate"] = p["date"].dt.to_period("Q").dt.start_time
    pq = p.groupby(["geolocation","qdate"],as_index=False)["target"].mean().rename(
        columns={"qdate":"date","target":"quarter_price"}
    )
    s = s.merge(pq, on=["geolocation","date"], how="left")

    g = s.groupby("geolocation", group_keys=False)
    s["area_lag1"] = g["area"].shift(1)
    s["area_lag4"] = g["area"].shift(4)
    s["yield_proxy_lag1"] = g["yield_proxy"].shift(1)
    s["yield_proxy_lag4"] = g["yield_proxy"].shift(4)
    s["quarter_price_lag1"] = g["quarter_price"].shift(1)
    supply_panels[c] = add_panel_lags(s, 4, [1,2,4,8])

# Price features: map agricultural context and lag it by one quarter (3 months).
for c in COMMODITIES:
    p = price_panels[c].sort_values(["geolocation","date"]).copy()
    s = supply_panels[c][["geolocation","date","target","area","yield_proxy"]].copy()
    s = s.rename(columns={"target":"quarter_supply","date":"qdate"})
    p["qdate"] = p["date"].dt.to_period("Q").dt.start_time
    p = p.merge(s, on=["geolocation","qdate"], how="left")
    p["quarter_supply_lag"] = p.groupby("geolocation")["quarter_supply"].shift(3)
    p["area_lag"] = p.groupby("geolocation")["area"].shift(3)
    p["yield_proxy_lag"] = p.groupby("geolocation")["yield_proxy"].shift(3)
    price_panels[c] = add_panel_lags(p, 12, [1,2,3,6,12,24])

for c,x in supply_panels.items():
    x.to_csv(ARTIFACT_ROOT / "prepared" / f"{c}_supply_features.csv", index=False)
for c,x in price_panels.items():
    x.to_csv(ARTIFACT_ROOT / "prepared" / f"{c}_price_features.csv", index=False)


## 7. Time-series model selection and recent rolling-origin validation

V3.3 separates **model selection** from **final evaluation**:

1. Reserve the most recent 4 quarters (supply) or 12 months (price) as an untouched final holdout.
2. On the older training period only, run dense expanding-window rolling validation over the most recent 12 quarters (supply) or 24 months (price), where data permits.
3. Select the candidate with the best robust rolling profile (median sMAPE, then 90th-percentile sMAPE/WAPE/MASE tie-breaks).
4. Evaluate that already-selected model once on the untouched final holdout.

Candidate family:
- seasonal naive
- HistGradientBoosting
- HistGradientBoosting with Poisson loss for nonnegative targets
- GradientBoosting with Huber loss for outlier robustness
- Random Forest

The single worst rolling fold is still reported for transparency, but the deployment gate uses the rolling median and 90th percentile so one isolated historical shock does not dominate the verdict.


In [ ]:
def feature_columns(kind, df):
    base = ["season_sin","season_cos","roll_mean_3","roll_std_3"]
    lags = sorted([c for c in df.columns if c.startswith("lag_")], key=lambda x: int(x.split("_")[1]))
    if kind == "supply":
        # Same-period area/yield_proxy are intentionally excluded: yield_proxy contains target_t.
        extra = [c for c in [
            "area_lag1","area_lag4","yield_proxy_lag1","yield_proxy_lag4","quarter_price_lag1"
        ] if c in df]
    else:
        extra = [c for c in ["quarter_supply_lag","area_lag","yield_proxy_lag"] if c in df]
    # Province is an explicit categorical feature instead of being inferred only from scale differences.
    return (["geolocation"] if "geolocation" in df.columns else []) + lags + base + extra


def build_ts_preprocessor(features):
    categorical = [c for c in features if c == "geolocation"]
    numeric = [c for c in features if c not in categorical]
    transformers = []
    if numeric:
        transformers.append(("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
        ]), numeric))
    if categorical:
        transformers.append(("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)


def make_candidate(name, **overrides):
    if name == "hist_gradient_boosting":
        params = dict(
            max_iter=300, learning_rate=0.04, max_leaf_nodes=15,
            l2_regularization=1.5, random_state=RANDOM_STATE
        )
        params.update(overrides)
        return HistGradientBoostingRegressor(**params)
    if name == "hist_gradient_boosting_poisson":
        params = dict(
            loss="poisson", max_iter=300, learning_rate=0.04, max_leaf_nodes=15,
            l2_regularization=2.0, random_state=RANDOM_STATE
        )
        params.update(overrides)
        return HistGradientBoostingRegressor(**params)
    if name == "gradient_boosting_huber":
        params = dict(
            loss="huber", n_estimators=300, learning_rate=0.035,
            max_depth=2, min_samples_leaf=4, subsample=0.85,
            random_state=RANDOM_STATE
        )
        params.update(overrides)
        return GradientBoostingRegressor(**params)
    if name == "random_forest":
        params = dict(
            n_estimators=500, max_depth=12, min_samples_leaf=2,
            max_features=0.85, n_jobs=-1, random_state=RANDOM_STATE
        )
        params.update(overrides)
        return RandomForestRegressor(**params)
    raise ValueError(name)


def build_ts_model(name, features, **overrides):
    return Pipeline([
        ("prep", build_ts_preprocessor(features)),
        ("model", make_candidate(name, **overrides)),
    ])


# --- V3.4: hyperparameter tuning grid --------------------------------------------------
# No candidate family was ever tuned in V3.3 -- every model used one hand-picked
# hyperparameter set. Each tunable family below is expanded into a small set of named
# variants (index 0 is always the original V3.3 default, so nothing regresses), and the
# existing rolling-origin race (rolling_selection_key) picks the best variant AND family
# in a single flat competition -- no separate search step, no extra risk of touching the
# final holdout during tuning.
TS_TUNABLE_FAMILIES = [
    "hist_gradient_boosting",
    "hist_gradient_boosting_poisson",
    "gradient_boosting_huber",
    "random_forest",
]

# Kept deliberately narrow (1 default + 1 alternate per family, 2 blend weights): rolling-
# origin validation refits a fresh model per fold, so grid width multiplies wall-clock time
# almost linearly (measured ~0.3-1.5s per fit on this machine for these estimator counts,
# independent of n_jobs -- the cost is real compute, not process-spawn overhead). This width
# targets roughly 2x V3.3's baseline training time rather than the 3-4x a fuller grid would cost.
TS_TUNING_GRID = {
    "hist_gradient_boosting": [
        {},
        {"learning_rate": 0.02, "max_leaf_nodes": 10, "l2_regularization": 3.0},
    ],
    "hist_gradient_boosting_poisson": [
        {},
        {"learning_rate": 0.02, "max_leaf_nodes": 10, "l2_regularization": 4.0},
    ],
    "gradient_boosting_huber": [
        {},
        {"learning_rate": 0.02, "max_depth": 2, "min_samples_leaf": 8, "subsample": 0.7},
    ],
    "random_forest": [
        {"n_jobs": 1},
        {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 4, "max_features": 0.6, "n_jobs": 1},
    ],
}

# Applied only to the winning candidate's final full-history refit (see step 3 in
# train_time_series_family) -- a single large fit is worth the multi-core startup cost.
RF_FINAL_OVERRIDES = {"n_jobs": -1}

# Blend weights tried against the single best-performing ML variant (see step 1b in
# train_time_series_family). ML-share weights only -- weight=1.0 (pure ML) is already
# covered by the untouched variant itself, so it is intentionally not repeated here.
BLEND_WEIGHTS = [0.4, 0.6]

# A blend candidate is only admitted to the race if its rolling p90 sMAPE (tail risk) is no
# more than this factor times its own base ingredient's p90 sMAPE. Blending should stabilize
# a model, not make its worst-case folds meaningfully worse -- see the guardrail note in
# train_time_series_family for the concrete failure this prevents.
BLEND_MAX_P90_RATIO = 1.2

# Half-life, in the series' own period unit (quarters for supply, months for price), for
# the recency sample-weighting applied only to the final artifact refit (never during
# candidate selection) -- targets the "wins on rolling CV, degrades on the most recent
# holdout" regime-shift symptom seen in Rice supply and Banana price.
RECENCY_HALF_LIFE_PERIODS = {"supply": 8, "price": 24}


def build_ts_candidate_specs():
    """variant_name -> {family, overrides, blend_weight}. `family` is always a name
    make_candidate() understands; `blend_weight`, when set, means the variant's final
    prediction is a weighted average of that family's prediction and the seasonal-naive
    value rather than the family's raw prediction."""
    specs = {"seasonal_naive": {"family": "seasonal_naive", "overrides": {}, "blend_weight": None}}
    for family in TS_TUNABLE_FAMILIES:
        grid = TS_TUNING_GRID.get(family, [{}])
        for i, overrides in enumerate(grid):
            name = family if i == 0 else f"{family}__t{i}"
            specs[name] = {"family": family, "overrides": overrides, "blend_weight": None}
    return specs


class SeasonalNaiveBlendRegressor:
    """Blends a fitted ML pipeline's prediction with the seasonal-naive value (the
    target's lag_{seasonal_period} feature, already present in `features`) to reduce
    final-holdout instability relative to the pure ML candidate.

    NOTE for deployment: this is a plain class defined in this notebook, not part of an
    importable package. If a blend variant wins and this artifact is later promoted to
    EF_agriwise (the final promotion cell), this class definition must be duplicated into
    an importable module there so joblib can unpickle it in the serving process.
    """
    def __init__(self, ml_pipeline, naive_col, ml_weight):
        self.ml_pipeline = ml_pipeline
        self.naive_col = naive_col
        self.ml_weight = ml_weight

    def predict(self, X):
        ml_pred = np.asarray(self.ml_pipeline.predict(X), dtype=float)
        naive_pred = np.asarray(X[self.naive_col], dtype=float)
        blended = self.ml_weight * ml_pred + (1 - self.ml_weight) * naive_pred
        return np.maximum(blended, 0.0)


def recency_sample_weight(date_series, half_life_periods):
    """Exponential recency weighting for the final artifact refit: the most recent period
    gets weight 1.0, and weight halves every `half_life_periods` periods further back."""
    dates = pd.to_datetime(pd.Series(date_series).to_numpy())
    uniq = np.sort(dates.unique())
    rank = {d: i for i, d in enumerate(uniq)}
    n = len(uniq)
    periods_back = np.array([n - 1 - rank[d] for d in dates.to_numpy()], dtype=float)
    return np.exp(-np.log(2) * periods_back / half_life_periods)
# -----------------------------------------------------------------------------------------


def chronological_split(df, holdout_periods):
    dates = np.array(sorted(df["date"].dropna().unique()))
    if len(dates) <= holdout_periods + 4:
        return None, None
    cutoff = dates[-holdout_periods]
    return df[df["date"] < cutoff].copy(), df[df["date"] >= cutoff].copy()


def seasonal_naive_predict(train, test, seasonal_period):
    hist = pd.concat([
        train[["geolocation","date","target"]],
        test[["geolocation","date","target"]].assign(target=np.nan)
    ]).sort_values(["geolocation","date"])
    hist["naive"] = hist.groupby("geolocation")["target"].shift(seasonal_period)
    return hist.loc[hist["date"].isin(test["date"]), ["geolocation","date","naive"]]


def evaluate_candidate(train, test, kind, family, seasonal_period, overrides=None, blend_weight=None):
    """If blend_weight is given, the returned prediction is a weighted average of `family`'s
    prediction (weight=blend_weight) and the seasonal-naive value (weight=1-blend_weight)."""
    overrides = overrides or {}
    feats = feature_columns(kind, train)
    required = f"lag_{seasonal_period}"
    tr = train.dropna(subset=["target", required]).copy()
    te = test.dropna(subset=["target", required]).copy()
    if len(tr) < 40 or len(te) == 0:
        return None

    naive = te[required].to_numpy()

    if family == "seasonal_naive":
        pred = naive.copy()
        model = None
    else:
        if family.endswith("poisson") and (tr["target"] < 0).any():
            return None
        use = [c for c in feats if c in tr.columns]
        model = build_ts_model(family, use, **overrides)
        model.fit(tr[use], tr["target"])
        ml_pred = model.predict(te[use])
        pred = (blend_weight * ml_pred + (1 - blend_weight) * naive) if blend_weight is not None else ml_pred
        # Supply and farmgate-price targets are nonnegative by definition.
        pred = np.maximum(pred, 0.0)

    metrics = regression_metrics(te["target"], pred, seasonal_naive=naive)
    return {"model":model, "metrics":metrics, "pred":pred, "test":te, "features":feats}


def rolling_validate(df, kind, family, seasonal_period, recent_periods, overrides=None, blend_weight=None):
    """Dense recent expanding-window validation. `df` must already exclude the final holdout."""
    dates = np.array(sorted(df["date"].dropna().unique()))
    min_train = max(seasonal_period * 3, 12)
    eligible = dates[min_train:]
    if len(eligible) == 0:
        return pd.DataFrame()
    test_dates = eligible[-min(recent_periods, len(eligible)):]

    rows = []
    for i, test_date in enumerate(test_dates, 1):
        tr = df[df["date"] < test_date]
        te = df[df["date"] == test_date]
        result = evaluate_candidate(tr, te, kind, family, seasonal_period, overrides=overrides, blend_weight=blend_weight)
        if result is None:
            continue
        rows.append({"split":i, "test_date":test_date, **result["metrics"]})
    return pd.DataFrame(rows)


def rolling_selection_key(rolling):
    if rolling is None or rolling.empty or "sMAPE" not in rolling:
        return (np.inf, np.inf, np.inf, np.inf, np.inf)
    sm = pd.to_numeric(rolling["sMAPE"], errors="coerce").dropna()
    wa = pd.to_numeric(rolling.get("WAPE", pd.Series(dtype=float)), errors="coerce").dropna()
    ma = pd.to_numeric(rolling.get("MASE", pd.Series(dtype=float)), errors="coerce").dropna()
    if sm.empty:
        return (np.inf, np.inf, np.inf, np.inf, np.inf)

    # V3.4 stability tiebreaker: penalize a candidate whose rolling sMAPE is trending worse
    # in the most recent half of folds vs. the earliest half, even if its overall median
    # looks fine -- this is what a good-on-CV/bad-on-final-holdout candidate looks like.
    degradation = 0.0
    if "test_date" in rolling:
        ordered = pd.to_numeric(rolling.sort_values("test_date")["sMAPE"], errors="coerce").dropna()
        if len(ordered) >= 4:
            half = len(ordered) // 2
            degradation = max(0.0, float(ordered.iloc[-half:].mean() - ordered.iloc[:half].mean()))

    return (
        float(sm.median()),
        float(sm.quantile(0.90)),
        float(wa.median()) if not wa.empty else np.inf,
        float(ma.median()) if not ma.empty else np.inf,
        degradation,
    )


def deployment_verdict(metrics, rolling, improvement, selected_name):
    sm = metrics.get("sMAPE", np.nan)
    wape = metrics.get("WAPE", np.nan)
    mase = metrics.get("MASE", np.nan)
    r2 = metrics.get("R2", np.nan)
    roll_med = rolling["sMAPE"].median() if not rolling.empty else np.nan
    roll_p90 = rolling["sMAPE"].quantile(.90) if not rolling.empty else np.nan

    if not np.isfinite(sm):
        return "INSUFFICIENT_DATA", "no valid holdout error"

    # A negative final R2 is an explicit caution even when percentage/absolute errors look acceptable.
    negative_r2 = np.isfinite(r2) and r2 < 0

    if selected_name == "seasonal_naive":
        # MASE=1 is tautological for the baseline itself, not evidence that a useful stable baseline failed.
        if sm <= 25 and (not np.isfinite(wape) or wape <= 20) and (not np.isfinite(roll_p90) or roll_p90 <= 40):
            if negative_r2:
                return "CAUTION", "seasonal baseline has acceptable error but negative holdout R2"
            return "PASS", "stable seasonal-naive baseline selected; ML candidates did not improve robust rolling validation"
        if sm <= 50 and (not np.isfinite(roll_med) or roll_med <= 50):
            return "CAUTION", "seasonal-naive baseline is the most robust candidate but error remains moderate"
        return "INSUFFICIENT_DATA", "forecast errors remain too high"

    mase_ok = not np.isfinite(mase) or mase < 1.0
    if sm <= 30 and (not np.isfinite(roll_med) or roll_med <= 35) and (not np.isfinite(roll_p90) or roll_p90 <= 50):
        if negative_r2:
            return "CAUTION", "absolute/percentage error is acceptable but negative holdout R2 indicates weak generalization"
        if not mase_ok:
            return "CAUTION", f"MASE={mase:.2f} indicates no improvement over the seasonal-naive baseline"
        return "PASS", ""
    if sm <= 50 and (not np.isfinite(roll_med) or roll_med <= 50):
        return "CAUTION", "moderate forecast error and/or rolling variability"
    return "INSUFFICIENT_DATA", "forecast errors remain too high"


def train_time_series_family(panels, kind):
    holdout_periods = 4 if kind == "supply" else 12
    seasonal_period = 4 if kind == "supply" else 12
    recent_periods = 12 if kind == "supply" else 24
    base_specs = build_ts_candidate_specs()

    all_metrics, all_roll, verdicts = [], [], []

    for commodity,df in panels.items():
        df = df.copy().dropna(subset=["target"])
        train,test = chronological_split(df, holdout_periods)
        if train is None:
            verdicts.append({"commodity":commodity,"kind":kind,"verdict":"INSUFFICIENT_DATA","reason":"too few periods"})
            continue

        specs = dict(base_specs)  # per-commodity copy; blend variants are added below

        # 1) Model selection occurs only inside the pre-holdout training period. Every
        # tunable family's hyperparameter variants race head-to-head with each other and
        # with seasonal_naive in one flat competition.
        candidate_rolls = {
            name: rolling_validate(
                train, kind, spec["family"], seasonal_period, recent_periods,
                overrides=spec["overrides"], blend_weight=spec["blend_weight"],
            )
            for name, spec in specs.items()
        }

        # 1b) Blend the single best-performing non-naive variant with the seasonal-naive
        # baseline at a few weights, and let those blends compete in the same race. A blend
        # is admitted to the race only if it doesn't amplify tail risk relative to its own
        # base ingredient (see BLEND_MAX_P90_RATIO below) -- blending is supposed to
        # stabilize the model, and rolling_selection_key ranks primarily on the MEDIAN, so an
        # unstable blend with a much worse p90 can still win purely on a slightly-better
        # median while being a strictly worse choice on every fold that actually matters. This
        # guardrail is what's needed after an earlier run showed exactly that failure mode for
        # Banana price: a blend with the best median won selection, but its rolling p90 sMAPE
        # was 58% worse than its own base model's, and on the untouched final holdout every
        # other candidate -- including the plain base model -- beat it.
        ml_names = [n for n,s in specs.items() if s["family"] != "seasonal_naive"]
        selectable_ml = [n for n in ml_names if candidate_rolls[n] is not None and not candidate_rolls[n].empty]
        naive_roll = candidate_rolls.get("seasonal_naive")
        if selectable_ml and naive_roll is not None and not naive_roll.empty:
            best_ml_name = min(selectable_ml, key=lambda n: rolling_selection_key(candidate_rolls[n]))
            best_ml_spec = specs[best_ml_name]
            base_p90 = rolling_selection_key(candidate_rolls[best_ml_name])[1]
            for w in BLEND_WEIGHTS:
                bname = f"blend_naive_{best_ml_name}_w{int(w*100)}"
                blend_roll = rolling_validate(
                    train, kind, best_ml_spec["family"], seasonal_period, recent_periods,
                    overrides=best_ml_spec["overrides"], blend_weight=w,
                )
                blend_p90 = rolling_selection_key(blend_roll)[1]
                tail_risk_ok = (not np.isfinite(base_p90)) or (not np.isfinite(blend_p90)) or (blend_p90 <= BLEND_MAX_P90_RATIO * base_p90)
                if blend_roll is not None and not blend_roll.empty and tail_risk_ok:
                    specs[bname] = {"family":best_ml_spec["family"], "overrides":best_ml_spec["overrides"], "blend_weight":w}
                    candidate_rolls[bname] = blend_roll

        selectable = [n for n,r in candidate_rolls.items() if r is not None and not r.empty]
        if not selectable:
            verdicts.append({"commodity":commodity,"kind":kind,"verdict":"INSUFFICIENT_DATA","reason":"no candidate could be rolling-validated"})
            continue
        selected_name = min(selectable, key=lambda n: rolling_selection_key(candidate_rolls[n]))

        # 2) Only after selection, evaluate all candidates on the untouched final holdout for transparent reporting.
        holdout_candidates = {}
        for name, spec in specs.items():
            res = evaluate_candidate(
                train, test, kind, spec["family"], seasonal_period,
                overrides=spec["overrides"], blend_weight=spec["blend_weight"],
            )
            if res is not None:
                holdout_candidates[name] = res

        if selected_name not in holdout_candidates:
            # Rare sparse-data fallback: choose the best rolling candidate that can also be evaluated on holdout.
            evaluable = [n for n in selectable if n in holdout_candidates]
            if not evaluable:
                verdicts.append({"commodity":commodity,"kind":kind,"verdict":"INSUFFICIENT_DATA","reason":"selected candidates have no valid final holdout"})
                continue
            selected_name = min(evaluable, key=lambda n: rolling_selection_key(candidate_rolls[n]))

        selected = holdout_candidates[selected_name]
        selected_spec = specs[selected_name]
        baseline = holdout_candidates.get("seasonal_naive")
        base_sm = baseline["metrics"].get("sMAPE",np.nan) if baseline else np.nan
        sel_sm = selected["metrics"].get("sMAPE",np.nan)
        improvement = ((base_sm-sel_sm)/base_sm*100) if np.isfinite(base_sm) and base_sm > 0 else np.nan
        rolling = candidate_rolls[selected_name]
        verdict,reason = deployment_verdict(selected["metrics"], rolling, improvement, selected_name)

        # 3) Refit the already-selected strategy on all valid history for artifact export.
        # A recency-weighted sample_weight is applied only here (never during selection) so
        # the deployed model favors the most recent regime without changing which candidate wins.
        feats = selected["features"]
        valid = df.dropna(subset=["target",f"lag_{seasonal_period}"]).copy()
        final_model = None
        is_blend = selected_spec["blend_weight"] is not None
        if selected_spec["family"] != "seasonal_naive":
            final_overrides = dict(selected_spec["overrides"])
            if selected_spec["family"] == "random_forest":
                final_overrides.update(RF_FINAL_OVERRIDES)
            base_final = build_ts_model(selected_spec["family"], feats, **final_overrides)
            sw = recency_sample_weight(valid["date"], RECENCY_HALF_LIFE_PERIODS[kind])
            base_final.fit(valid[feats], valid["target"], model__sample_weight=sw)
            final_model = SeasonalNaiveBlendRegressor(base_final, f"lag_{seasonal_period}", selected_spec["blend_weight"]) if is_blend else base_final

        roll_summary = {
            "median_sMAPE": float(rolling["sMAPE"].median()) if not rolling.empty else np.nan,
            "p90_sMAPE": float(rolling["sMAPE"].quantile(.90)) if not rolling.empty else np.nan,
            "worst_sMAPE": float(rolling["sMAPE"].max()) if not rolling.empty else np.nan,
            "median_WAPE": float(rolling["WAPE"].median()) if not rolling.empty and "WAPE" in rolling else np.nan,
            "median_MASE": float(rolling["MASE"].median()) if not rolling.empty and "MASE" in rolling else np.nan,
            "folds": int(len(rolling)),
        }

        limitations = [
            f"{kind.capitalize()} model selected by recent rolling-origin validation on the pre-holdout period; final holdout was not used for model selection.",
            "Same-period target-derived agronomic features are excluded from supply predictors.",
        ]
        if is_blend:
            limitations.append(
                "Final prediction blends the ML model with the seasonal-naive baseline (see blend_base_family/blend_weight). "
                "If promoted to EF_agriwise, SeasonalNaiveBlendRegressor must be importable wherever this artifact is unpickled."
            )

        artifact = {
            "schema_version":"3.4",
            "created_utc":datetime.utcnow().isoformat()+"Z",
            "commodity":COMMODITY_TITLE_CASE.get(commodity, commodity),
            "model_id":f"{commodity}-{kind}-v3",
            "kind":kind,
            "strategy":selected_name,
            "model":final_model,
            "features":feats,
            "seasonal_period":seasonal_period,
            "metrics":selected["metrics"],
            "baseline_metrics":baseline["metrics"] if baseline else {},
            "improvement_vs_naive_pct":improvement,
            "rolling_selection_summary":roll_summary,
            "verdict":verdict,
            "reason":reason,
            "limitations":limitations,
            "hyperparameter_overrides": selected_spec["overrides"],
            "blend_base_family": selected_spec["family"] if is_blend else None,
            "blend_weight": selected_spec["blend_weight"],
            "history_tail":valid.groupby("geolocation").tail(seasonal_period*2)[
                ["geolocation","date","target"]
            ].to_dict("records"),
        }
        joblib.dump(artifact, ARTIFACT_ROOT / kind / f"{commodity}.joblib")

        for name,res in holdout_candidates.items():
            all_metrics.append({
                "commodity":commodity,"kind":kind,"model":name,
                "selected_by_rolling_cv":name == selected_name,
                **res["metrics"],
            })
        for name,r in candidate_rolls.items():
            if r is not None and not r.empty:
                rr = r.copy()
                rr["commodity"] = commodity
                rr["kind"] = kind
                rr["model"] = name
                rr["selected_by_rolling_cv"] = name == selected_name
                all_roll.append(rr)

        verdicts.append({
            "commodity":commodity,"kind":kind,"selected_model":selected_name,
            **selected["metrics"],
            "baseline_sMAPE":base_sm,
            "improvement_vs_naive_pct":improvement,
            "rolling_median_sMAPE":roll_summary["median_sMAPE"],
            "rolling_p90_sMAPE":roll_summary["p90_sMAPE"],
            "rolling_worst_sMAPE":roll_summary["worst_sMAPE"],
            "rolling_median_WAPE":roll_summary["median_WAPE"],
            "rolling_median_MASE":roll_summary["median_MASE"],
            "rolling_folds":roll_summary["folds"],
            "selection_basis":"recent rolling-origin validation over a tuned hyperparameter grid; final holdout reserved",
            "verdict":verdict,"reason":reason,
        })
        print(kind, commodity, "=>", selected_name, verdict, reason)

    return (
        pd.DataFrame(all_metrics),
        pd.concat(all_roll,ignore_index=True) if all_roll else pd.DataFrame(),
        pd.DataFrame(verdicts),
    )


In [9]:

supply_metrics, supply_rolling, supply_verdicts = train_time_series_family(supply_panels, "supply")
price_metrics, price_rolling, price_verdicts = train_time_series_family(price_panels, "price")

supply_metrics.to_csv(ARTIFACT_ROOT / "reports" / "supply_model_metrics.csv", index=False)
supply_rolling.to_csv(ARTIFACT_ROOT / "reports" / "supply_rolling_validation.csv", index=False)
supply_verdicts.to_csv(ARTIFACT_ROOT / "reports" / "supply_deployment_verdicts.csv", index=False)

price_metrics.to_csv(ARTIFACT_ROOT / "reports" / "price_model_metrics.csv", index=False)
price_rolling.to_csv(ARTIFACT_ROOT / "reports" / "price_rolling_validation.csv", index=False)
price_verdicts.to_csv(ARTIFACT_ROOT / "reports" / "price_deployment_verdicts.csv", index=False)

display(supply_verdicts)
display(price_verdicts)


supply rice => gradient_boosting_huber CAUTION moderate forecast error and/or rolling variability
supply tomato => seasonal_naive CAUTION seasonal-naive baseline is the most robust candidate but error remains moderate
supply red_onion => seasonal_naive INSUFFICIENT_DATA forecast errors remain too high
supply banana => random_forest PASS 
price rice => random_forest PASS 
price tomato => hist_gradient_boosting CAUTION moderate forecast error and/or rolling variability
price banana => random_forest CAUTION absolute/percentage error is acceptable but negative holdout R2 indicates weak generalization


,commodity,kind,selected_model,MAE,MSE,RMSE,R2,MAPE,sMAPE,WAPE,...,improvement_vs_naive_pct,rolling_median_sMAPE,rolling_p90_sMAPE,rolling_worst_sMAPE,rolling_median_WAPE,rolling_median_MASE,rolling_folds,selection_basis,verdict,reason
0,rice,supply,gradient_boosting_huber,2657.071047,2.538374e+07,5038.227611,0.883443,45.436608,33.844399,22.184703,...,-49.577894,24.058924,51.087093,52.126097,23.899322,1.029933,12,recent rolling-origin validation; final holdou...,CAUTION,moderate forecast error and/or rolling variabi...
1,tomato,supply,seasonal_naive,102.245500,7.444939e+04,272.854163,0.968795,23.868586,28.999335,13.447941,...,0.000000,21.597907,51.556386,82.562693,12.062414,1.000000,12,recent rolling-origin validation; final holdou...,CAUTION,seasonal-naive baseline is the most robust can...
2,red_onion,supply,seasonal_naive,1.261071,8.569416e+00,2.927357,0.515448,1244.634380,64.745796,50.477470,...,0.000000,82.069299,192.204720,200.000000,49.375720,1.000000,12,recent rolling-origin validation; final holdou...,INSUFFICIENT_DATA,forecast errors remain too high
3,banana,supply,random_forest,571.611678,1.264655e+06,1124.568646,0.936465,15.779644,20.025857,12.855987,...,-5.579435,10.854024,18.708958,21.894873,10.072645,0.878733,12,recent rolling-origin validation; final holdou...,PASS,


,commodity,kind,selected_model,MAE,MSE,RMSE,R2,MAPE,sMAPE,WAPE,...,improvement_vs_naive_pct,rolling_median_sMAPE,rolling_p90_sMAPE,rolling_worst_sMAPE,rolling_median_WAPE,rolling_median_MASE,rolling_folds,selection_basis,verdict,reason
0,rice,price,random_forest,1.990229,7.362220,2.713341,0.639951,10.922603,11.146308,11.291915,...,68.411807,7.132804,12.065455,18.525786,7.266526,0.294368,24.0,recent rolling-origin validation; final holdou...,PASS,
1,tomato,price,hist_gradient_boosting,17.611458,521.653905,22.839744,0.039344,45.731863,37.479223,39.791112,...,21.576616,42.740436,79.397476,114.144740,39.931969,0.742080,24.0,recent rolling-origin validation; final holdou...,CAUTION,moderate forecast error and/or rolling variabi...
2,red_onion,price,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INSUFFICIENT_DATA,no candidate could be rolling-validated
3,banana,price,random_forest,3.827389,28.014191,5.292843,-0.085015,15.275523,14.533540,14.929451,...,30.377330,15.315603,22.577078,38.386793,14.286784,0.686672,24.0,recent rolling-origin validation; final holdou...,CAUTION,absolute/percentage error is acceptable but ne...


## 8. Load FIES 2023 household-summary data and valueset

In [10]:

fies_summary_path = next(iter(lfs_root.rglob("FIES-LFS-houseHoldSummary.CSV")), None)
fies_xlsx_path = next(iter(lfs_root.rglob("fies_lfs.xlsx")), None)

assert fies_summary_path is not None, "FIES household summary not found."

fies = pd.read_csv(fies_summary_path, low_memory=False)
fies["PROVINCE_NAME"] = pd.to_numeric(fies["RPROV"],errors="coerce").map(FIES_PROVINCE_MAP)
fies_cal = fies[fies["PROVINCE_NAME"].notna()].copy()

print("FIES national rows:", len(fies))
print("CALABARZON rows:", len(fies_cal))
print(fies_cal["PROVINCE_NAME"].value_counts())

target_audit = []
for target in ["BREAD","VEG","FRUIT","FRUIT_VEG"]:
    y = pd.to_numeric(fies_cal[target],errors="coerce")
    target_audit.append({
        "target":target,
        "n":int(y.notna().sum()),
        "mean":y.mean(),
        "median":y.median(),
        "p95":y.quantile(.95),
        "zero_rate_pct":float(np.isclose(y.fillna(np.nan),0).mean()*100),
    })
target_audit_df = pd.DataFrame(target_audit)
display(target_audit_df)
target_audit_df.to_csv(ARTIFACT_ROOT / "reports" / "fies_target_audit.csv", index=False)


FIES national rows: 163268
CALABARZON rows: 8156
PROVINCE_NAME
Quezon      2413
Batangas    1482
Laguna      1479
Rizal       1447
Cavite      1335
Name: count, dtype: int64


,target,n,mean,median,p95,zero_rate_pct
0,BREAD,8156,26539.274440,24207.00,50349.500,0.036783
1,VEG,8156,7502.590002,6203.00,17179.375,0.478176
2,FRUIT,8156,4910.714766,3649.55,12922.500,0.220696
3,FRUIT_VEG,8156,780.054620,380.00,2886.750,23.001471


## 9. FIES demand-proxy estimation

This is **not** a time-series forecast.

The household model tests whether socioeconomic household attributes can estimate the FIES expenditure-category target and produces survey-weighted province-level proxy estimates.

### V3.3 validation design

- A stratified 25% household holdout is reserved using **province + target quantile** strata.
- Candidate selection happens only inside the remaining 75% using 3-fold stratified cross-validation.
- The selected family is then evaluated once on the untouched 25% holdout.
- Province leave-one-out remains a separate geographic-generalization audit.

### Leakage controls

Excluded from predictors:
- `SEQ_NO` and identifiers
- `BREAD`, `VEG`, `FRUIT`
- `FOOD_HOME`, `FOOD`, `TOTEX`, and other totals that directly contain/reflect the food target
- target-family components

`RFACT` is used as a **survey training/aggregation weight**, not as a household predictor.


In [ ]:
available_num = [c for c in DEMAND_NUMERIC_FEATURES if c in fies_cal.columns]
available_cat = [c for c in DEMAND_CATEGORICAL_FEATURES if c in fies_cal.columns]


def build_demand_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute",SimpleImputer(strategy="median"))]), available_num),
        ("cat", Pipeline([
            ("impute",SimpleImputer(strategy="most_frequent")),
            ("onehot",OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), available_cat),
    ], remainder="drop", sparse_threshold=0)


def build_demand_pipeline_gb(**overrides):
    params = dict(n_estimators=300, learning_rate=.04, max_depth=3, min_samples_leaf=6, random_state=RANDOM_STATE)
    params.update(overrides)
    return Pipeline([
        ("prep", build_demand_preprocessor()),
        ("model", GradientBoostingRegressor(**params)),
    ])


def build_demand_pipeline_huber(**overrides):
    params = dict(
        loss="huber", n_estimators=350, learning_rate=.035, max_depth=2,
        min_samples_leaf=8, subsample=.85, random_state=RANDOM_STATE
    )
    params.update(overrides)
    return Pipeline([
        ("prep", build_demand_preprocessor()),
        ("model", GradientBoostingRegressor(**params)),
    ])


def build_demand_pipeline_log_gb(**overrides):
    # Built-in TransformedTargetRegressor keeps artifact predict() semantics unchanged.
    base = build_demand_pipeline_gb(**overrides)
    return TransformedTargetRegressor(
        regressor=base,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


def build_demand_pipeline_xgb(**overrides):
    params = dict(DEMAND_XGB_PARAMETERS)
    params.update(overrides)
    return Pipeline([
        ("prep", build_demand_preprocessor()),
        ("model", XGBRegressor(**params)),
    ])


def build_demand_pipeline_poisson(**overrides):
    params = dict(
        loss="poisson", max_iter=350, learning_rate=.04,
        max_leaf_nodes=15, min_samples_leaf=15,
        l2_regularization=2.0, random_state=RANDOM_STATE,
    )
    params.update(overrides)
    return Pipeline([
        ("prep", build_demand_preprocessor()),
        ("model", HistGradientBoostingRegressor(**params)),
    ])


def fit_weighted(estimator, X, y, w):
    # All current candidates expose a nested Pipeline whose final step is named `model`.
    # TransformedTargetRegressor forwards fit params to its wrapped Pipeline.
    estimator.fit(X, y, model__sample_weight=np.asarray(w, dtype=float))
    return estimator


def weighted_mean(y,w):
    y=np.asarray(y,float); w=np.asarray(w,float)
    m=np.isfinite(y)&np.isfinite(w)&(w>0)
    return np.average(y[m],weights=w[m]) if m.any() else np.nan


def province_mean_naive(train_y, train_prov, test_prov):
    means = train_y.groupby(train_prov).mean()
    overall = float(np.nanmean(train_y.to_numpy()))
    return test_prov.map(means).fillna(overall).to_numpy()


def make_demand_strata(y, province, bins=5):
    # rank(method='first') avoids qcut duplicate-edge failures when many households share values/zeros.
    y_rank = pd.Series(y, index=province.index).rank(method="first")
    q = pd.qcut(y_rank, q=min(bins, max(2, int(len(y_rank) / 200))), labels=False, duplicates="drop")
    strata = province.astype(str) + "__q" + q.astype(str)
    # Safety fallback if any combined stratum is too small for splitting.
    counts = strata.value_counts()
    if (counts < 3).any():
        return province.astype(str)
    return strata


def demand_cv_score(factory, X, y, weights, province, strata, n_splits=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    rows=[]
    for fold,(tr_idx,va_idx) in enumerate(skf.split(X, strata),1):
        model=factory()
        fit_weighted(model, X.iloc[tr_idx], y.iloc[tr_idx], weights.iloc[tr_idx])
        pred=np.maximum(model.predict(X.iloc[va_idx]), 0.0)
        prov_tr = province.iloc[tr_idx]
        prov_va = province.iloc[va_idx]
        naive = province_mean_naive(y.iloc[tr_idx], prov_tr, prov_va)
        wm=weighted_regression_metrics(y.iloc[va_idx], pred, weights.iloc[va_idx])
        um=regression_metrics(y.iloc[va_idx], pred, seasonal_naive=naive)
        rows.append({
            "fold":fold,
            "weighted_R2":wm.get("r2",np.nan),
            "WAPE":um.get("WAPE",np.nan),
            "sMAPE":um.get("sMAPE",np.nan),
            "MASE":um.get("MASE",np.nan),
        })
    cv=pd.DataFrame(rows)
    return {
        "folds":cv.to_dict("records"),
        "median_weighted_R2":float(cv["weighted_R2"].median()) if not cv.empty else np.nan,
        "median_WAPE":float(cv["WAPE"].median()) if not cv.empty else np.nan,
        "p90_WAPE":float(cv["WAPE"].quantile(.90)) if not cv.empty else np.nan,
        "median_sMAPE":float(cv["sMAPE"].median()) if not cv.empty else np.nan,
        "median_MASE":float(cv["MASE"].median()) if not cv.empty else np.nan,
    }


def demand_cv_key(cv):
    r2=cv.get("median_weighted_R2",np.nan)
    wape=cv.get("median_WAPE",np.nan)
    p90=cv.get("p90_WAPE",np.nan)
    mase=cv.get("median_MASE",np.nan)
    return (
        -(r2 if np.isfinite(r2) else -np.inf),
        wape if np.isfinite(wape) else np.inf,
        p90 if np.isfinite(p90) else np.inf,
        mase if np.isfinite(mase) else np.inf,
    )


# --- V3.4: hyperparameter tuning grid --------------------------------------------------
# No candidate family was ever tuned in V3.3 -- "XGBRegressor_regularized" was one
# hand-picked override point, not a search. Each tunable family below is expanded into a
# small set of named variants (index 0 is always the original V3.3 default, so nothing
# regresses), and the existing stratified-CV race (demand_cv_score/demand_cv_key) picks
# the best variant AND family in a single flat competition, exactly as before.
DEMAND_BASE_FACTORIES = {
    "GradientBoostingRegressor": build_demand_pipeline_gb,
    "GradientBoostingRegressor_huber": build_demand_pipeline_huber,
    "GradientBoostingRegressor_log1p": build_demand_pipeline_log_gb,
    "XGBRegressor": build_demand_pipeline_xgb,
    "HistGradientBoostingRegressor_poisson": build_demand_pipeline_poisson,
}

# Kept deliberately narrow (1 default + 1 alternate per family): each grid point is refit
# 3x (folds) per target, and FIES has thousands of rows with some families using hundreds
# of estimators, so grid width multiplies wall-clock time almost linearly. This width
# targets roughly 2x V3.3's baseline demand-training time rather than a much larger grid.
DEMAND_TUNING_GRID = {
    "GradientBoostingRegressor": [
        {},
        {"learning_rate": 0.02, "max_depth": 2, "min_samples_leaf": 12},
    ],
    "GradientBoostingRegressor_huber": [
        {},
        {"learning_rate": 0.02, "max_depth": 2, "min_samples_leaf": 15},
    ],
    "XGBRegressor": [
        {},
        # This combo is the exact V3.3 "XGBRegressor_regularized" hand-picked override.
        {"max_depth": 2, "min_child_weight": 20, "reg_lambda": 10.0, "learning_rate": 0.01},
    ],
    "HistGradientBoostingRegressor_poisson": [
        {},
        {"learning_rate": 0.02, "max_leaf_nodes": 10, "min_samples_leaf": 25, "l2_regularization": 4.0},
    ],
}


def build_demand_candidate_factories():
    """Expands each tunable family into named hyperparameter variants so the existing
    stratified-CV race picks the best combination and family together in one flat
    competition. GradientBoostingRegressor_log1p reuses the plain-GB grid since it wraps
    the same underlying GradientBoostingRegressor via TransformedTargetRegressor."""
    factories = {}
    for family, base_factory in DEMAND_BASE_FACTORIES.items():
        grid_key = "GradientBoostingRegressor" if family == "GradientBoostingRegressor_log1p" else family
        grid = DEMAND_TUNING_GRID.get(grid_key, [{}])
        for i, overrides in enumerate(grid):
            name = family if i == 0 else f"{family}__t{i}"
            factories[name] = (lambda bf=base_factory, ov=overrides: bf(**ov))
    return factories
# -----------------------------------------------------------------------------------------


# Broad-category proxy specificity, mirrored in commodity_demand_registry below.
DEMAND_SPECIFICITY = {
    "rice": "category_proxy",
    "vegetable_shared": "shared_broad_category",
    "banana": "category_proxy",
}

demand_model_rows=[]
demand_province_rows=[]
demand_artifacts={}

# Train only one VEG model and share it for tomato/red_onion to avoid pretending FIES distinguishes them.
target_models = {"BREAD":"rice", "VEG":"vegetable_shared", "FRUIT":"banana"}

for target, artifact_name in target_models.items():
    cols = available_num + available_cat
    assert target not in cols and "RFACT" not in cols, f"{artifact_name}: target or survey weight leaked into features"

    X = fies_cal[cols].copy()
    X["RPROV"] = X["RPROV"].astype(str)
    y = pd.to_numeric(fies_cal[target],errors="coerce")
    valid = y.notna()
    X,y = X.loc[valid],y.loc[valid]
    weights = pd.to_numeric(fies_cal.loc[valid,"RFACT"],errors="coerce").fillna(1.0)
    province = fies_cal.loc[valid,"PROVINCE_NAME"]

    strata = make_demand_strata(y, province)
    tr_idx, te_idx = train_test_split(
        np.arange(len(X)), test_size=.25, random_state=RANDOM_STATE, stratify=strata.to_numpy()
    )
    Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
    ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]
    wtr, wte = weights.iloc[tr_idx], weights.iloc[te_idx]
    prov_tr, prov_te = province.iloc[tr_idx], province.iloc[te_idx]
    naive_te = province_mean_naive(ytr, prov_tr, prov_te)
    cv_strata = strata.iloc[tr_idx].reset_index(drop=True)

    candidate_factories = build_demand_candidate_factories()

    # Candidate selection is based only on CV inside the training partition.
    cv_results={}
    Xtr_cv=Xtr.reset_index(drop=True)
    ytr_cv=ytr.reset_index(drop=True)
    wtr_cv=wtr.reset_index(drop=True)
    prov_tr_cv=prov_tr.reset_index(drop=True)
    for name,factory in candidate_factories.items():
        cv_results[name]=demand_cv_score(factory, Xtr_cv, ytr_cv, wtr_cv, prov_tr_cv, cv_strata, n_splits=3)

    best_name=min(cv_results, key=lambda n: demand_cv_key(cv_results[n]))
    best_factory=candidate_factories[best_name]

    # Final holdout is evaluated only after the winning family is fixed.
    holdout_candidates={}
    for name,factory in candidate_factories.items():
        model=factory()
        fit_weighted(model, Xtr, ytr, wtr)
        pred=np.maximum(model.predict(Xte),0.0)
        holdout_candidates[name]={
            "factory":factory,
            "pred":pred,
            "weighted":weighted_regression_metrics(yte,pred,wte),
            "unweighted":regression_metrics(yte,pred,seasonal_naive=naive_te),
        }

    best=holdout_candidates[best_name]
    m_weighted=best["weighted"]
    m_unweighted=best["unweighted"]
    actual_w=weighted_mean(yte,wte)
    pred_w=weighted_mean(best["pred"],wte)
    aggregate_error_pct=(pred_w-actual_w)/actual_w*100 if actual_w else np.nan
    verdict, verdict_reason = demand_verdict(m_weighted.get("r2",np.nan),m_unweighted.get("WAPE",np.nan))

    # Province leave-one-out audit: geographic generalization of the already-selected family.
    province_scores=[]
    for prov in CALABARZON:
        test_mask=province.eq(prov).to_numpy()
        if test_mask.sum()==0 or (~test_mask).sum()<100:
            continue
        gm=best_factory()
        fit_weighted(gm,X.loc[~test_mask],y.loc[~test_mask],weights.loc[~test_mask])
        gp=np.maximum(gm.predict(X.loc[test_mask]),0.0)
        province_scores.append({
            "province":prov,
            "R2":r2_score(y.loc[test_mask],gp) if y.loc[test_mask].std()>0 else np.nan,
            "MAE":mean_absolute_error(y.loc[test_mask],gp),
        })

    # Refit the winning family on all CALABARZON households, weighted.
    final_model=best_factory()
    fit_weighted(final_model,X,y,weights)
    all_pred=np.maximum(final_model.predict(X),0.0)
    assert np.isfinite(all_pred).all(),f"{artifact_name}: non-finite predictions after full refit"

    specificity=DEMAND_SPECIFICITY[artifact_name]
    limitations=[
        "FIES expenditure-category proxy, not physical commodity demand.",
        "Cross-sectional 2023 estimator; not an observed quarterly demand forecast.",
        "Model family and hyperparameters selected by stratified cross-validation inside the training partition; final holdout reserved for reporting.",
    ]
    if specificity=="shared_broad_category":
        limitations.append("Single VEG model shared across Tomato and Red Onion; FIES does not distinguish these commodities.")

    tuning_note="selected, together with its hyperparameters, by 3-fold province+target-quantile stratified CV over a small per-family grid on the 75% training partition; final 25% holdout was untouched until selection"
    art={
        "schema_version":"3.4",
        "kind":"demand_proxy_estimator",
        "target":target,
        "artifact_name":artifact_name,
        "model_id":f"{artifact_name}-demand-v3",
        "model":final_model,
        "selected_model":best_name,
        "candidate_race":{
            n:{"cv":cv_results[n],"weighted":c["weighted"],"unweighted":c["unweighted"]}
            for n,c in holdout_candidates.items()
        },
        "tuning_note":tuning_note,
        "features":cols,
        "weighted_metrics":m_weighted,
        "unweighted_metrics":m_unweighted,
        "weighted_aggregate_error_pct":aggregate_error_pct,
        "province_holdout":province_scores,
        "verdict":verdict,
        "verdict_reason":verdict_reason,
        "commodity_specificity":specificity,
        "dataset_counts":{
            "train_households":int(len(tr_idx)),
            "heldout_households":int(len(te_idx)),
            "provinces":sorted(province.unique().tolist()),
        },
        "limitations":limitations,
    }
    joblib.dump(art,ARTIFACT_ROOT/"demand"/f"{artifact_name}.joblib")
    demand_artifacts[artifact_name]=art

    demand_model_rows.append({
        "target":target,"artifact":artifact_name,"selected_model":best_name,
        "CV_median_weighted_R2":cv_results[best_name].get("median_weighted_R2",np.nan),
        "CV_median_WAPE":cv_results[best_name].get("median_WAPE",np.nan),
        "R2":m_weighted.get("r2",np.nan),"MAE":m_weighted.get("mae",np.nan),"RMSE":m_weighted.get("rmse",np.nan),
        "unweighted_R2":m_unweighted.get("R2",np.nan),
        "WAPE":m_unweighted.get("WAPE",np.nan),"sMAPE":m_unweighted.get("sMAPE",np.nan),
        "MAPE":m_unweighted.get("MAPE",np.nan),"MASE":m_unweighted.get("MASE",np.nan),
        "Bias":m_unweighted.get("Bias",np.nan),"N":m_unweighted.get("N",np.nan),
        "ZeroRatePct":m_unweighted.get("ZeroRatePct",np.nan),"NonZeroN":m_unweighted.get("NonZeroN",np.nan),
        "weighted_aggregate_error_pct":aggregate_error_pct,
        "province_holdout_mean_MAE":np.mean([r["MAE"] for r in province_scores]) if province_scores else np.nan,
        "province_holdout_mean_R2":np.nanmean([r["R2"] for r in province_scores]) if province_scores else np.nan,
        "verdict":verdict,"verdict_reason":verdict_reason,
    })

    tmp=fies_cal.loc[valid,["PROVINCE_NAME","RFACT"]].copy()
    tmp["actual"]=y.to_numpy()
    tmp["predicted"]=all_pred
    for prov,g in tmp.groupby("PROVINCE_NAME"):
        demand_province_rows.append({
            "target":target,
            "province":prov,
            "actual_weighted_mean":weighted_mean(g["actual"],g["RFACT"]),
            "predicted_weighted_mean":weighted_mean(g["predicted"],g["RFACT"]),
            "households":len(g),
            "sum_weight":g["RFACT"].sum(),
        })

    print(target,"=>",best_name,verdict,"| CV weighted R2:",round(cv_results[best_name].get("median_weighted_R2",float("nan")),3),
          "| final holdout weighted R2:",round(m_weighted.get("r2",float("nan")),3),
          "| WAPE:",round(m_unweighted.get("WAPE",float("nan")),1))

demand_model_report=pd.DataFrame(demand_model_rows)
demand_province_baseline=pd.DataFrame(demand_province_rows)

display(demand_model_report)
display(demand_province_baseline)

demand_model_report.to_csv(ARTIFACT_ROOT/"reports"/"demand_proxy_model_metrics.csv",index=False)
demand_province_baseline.to_csv(ARTIFACT_ROOT/"prepared"/"demand_proxy_province_baseline.csv",index=False)


## 9b. Demand-proxy validation reports

Per-target GradientBoosting-vs-XGBoost comparison (weighted + unweighted metrics), MASE against a province-mean naive baseline, the `weighted_aggregate_error_pct` formula spelled out, province leave-one-out results, and an honest tiered verdict (`USABLE_PROXY` / `INDICATIVE_PROXY` / `INSUFFICIENT_DATA`) -- written to `reports/{artifact}_validation_report.md` per target and consolidated into `reports/demand_validation_summary.csv`.


In [12]:

DEMAND_REPORT_LABELS = {
    "rice": "Rice (via BREAD proxy)",
    "vegetable_shared": "Vegetable -- Tomato & Red Onion (via VEG proxy)",
    "banana": "Banana (via FRUIT proxy)",
}

def _fmt(v, spec="{:.3f}"):
    return spec.format(v) if v is not None and np.isfinite(v) else "n/a"

demand_validation_rows=[]

for artifact_name, art in demand_artifacts.items():
    label = DEMAND_REPORT_LABELS.get(artifact_name, artifact_name)
    wm = art["weighted_metrics"]
    um = art["unweighted_metrics"]
    counts = art["dataset_counts"]

    # Sanity checks -- fail loud rather than silently shipping a bad artifact.
    assert set(art["features"]).isdisjoint({art["target"], "RFACT"}), \
        f"{artifact_name}: target or survey weight leaked into features"
    assert wm.get("nonfinite_predictions", 0) == 0, f"{artifact_name}: non-finite predictions in holdout"
    assert um.get("N", 0) > 0, f"{artifact_name}: empty holdout set"

    lines = []
    lines.append(f"# Demand-proxy validation report -- {label}")
    lines.append("")
    lines.append(
        f"Artifact: `demand/{artifact_name}.joblib` | Target column: `{art['target']}` | "
        f"Model selected: **{art['selected_model']}**"
    )
    lines.append("")
    lines.append("## Dataset")
    lines.append(f"- Training households: {counts['train_households']}")
    lines.append(f"- Held-out households: {counts['heldout_households']}")
    lines.append(f"- Provinces covered: {', '.join(counts['provinces'])}")
    lines.append("")
    lines.append("## Candidate comparison (weighted metrics, sample_weight = RFACT)")
    lines.append("")
    lines.append("| Candidate | CV median weighted R2 | CV median WAPE | final-holdout weighted R2 | weighted MAE | weighted RMSE |")
    lines.append("|---|---|---|---|---|---|")
    for cname, cres in art["candidate_race"].items():
        cw = cres["weighted"]
        marker = " **(selected)**" if cname == art["selected_model"] else ""
        cv = cres.get("cv", {})
        lines.append(
            f"| {cname}{marker} | {_fmt(cv.get('median_weighted_R2'))} | "
            f"{_fmt(cv.get('median_WAPE'),'{:.1f}')}% | {_fmt(cw.get('r2'))} | "
            f"{_fmt(cw.get('mae'),'{:.1f}')} | {_fmt(cw.get('rmse'),'{:.1f}')} |"
        )
    lines.append("")
    lines.append(f"Tuning note: {art['tuning_note']}")
    lines.append("")
    lines.append("## Selected model -- full metric set")
    lines.append("")
    lines.append("| Metric | Weighted (survey-representative) | Unweighted (raw holdout) |")
    lines.append("|---|---|---|")
    lines.append(f"| R2 | {_fmt(wm.get('r2'))} | {_fmt(um.get('R2'))} |")
    lines.append(f"| MAE | {_fmt(wm.get('mae'),'{:.1f}')} | {_fmt(um.get('MAE'),'{:.1f}')} |")
    lines.append(f"| RMSE | {_fmt(wm.get('rmse'),'{:.1f}')} | {_fmt(um.get('RMSE'),'{:.1f}')} |")
    lines.append(f"| WAPE | -- | {_fmt(um.get('WAPE'),'{:.2f}')}% |")
    lines.append(f"| sMAPE | -- | {_fmt(um.get('sMAPE'),'{:.2f}')}% |")
    lines.append(f"| MAPE | -- | {_fmt(um.get('MAPE'),'{:.2f}')}% |")
    lines.append(f"| MASE (vs. province-mean naive) | -- | {_fmt(um.get('MASE'),'{:.3f}')} |")
    lines.append(f"| Bias | -- | {_fmt(um.get('Bias'),'{:.1f}')} |")
    lines.append("")
    lines.append(
        "MASE is computed against a **province-mean naive baseline** (predict each "
        "held-out household's target as the training-set mean for its province), not a "
        "seasonal-lag naive, because FIES 2023 is cross-sectional with no time axis. "
        "MASE < 1 means the model beats that province-mean baseline; MASE >= 1 means it does not."
    )
    lines.append("")
    lines.append("## Population-level bias check")
    lines.append("")
    lines.append(
        f"`weighted_aggregate_error_pct` = {_fmt(art['weighted_aggregate_error_pct'],'{:.2f}')}%. "
        "Formula: `(weighted_mean(prediction, RFACT) - weighted_mean(actual, RFACT)) / "
        "weighted_mean(actual, RFACT) * 100`, computed on the holdout set only. This is a "
        "population-level bias check -- overestimates and underestimates at the household "
        "level can cancel out here -- and is not a substitute for the per-household WAPE/MASE above."
    )
    lines.append("")
    lines.append("## Province leave-one-out holdout")
    lines.append("")
    lines.append("| Province | R2 | MAE |")
    lines.append("|---|---|---|")
    for row in art["province_holdout"]:
        lines.append(f"| {row['province']} | {_fmt(row['R2'])} | {_fmt(row['MAE'],'{:.1f}')} |")
    lines.append("")
    lines.append("## Proxy-mapping caveat")
    lines.append("")
    if art["commodity_specificity"] == "shared_broad_category":
        lines.append(
            f"`{art['target']}` is a broad FIES vegetable-expenditure category shared by Tomato "
            "and Red Onion. This model does not and cannot distinguish between the two "
            "commodities; it is deployed identically under both."
        )
    else:
        commodity_word = label.split(" (")[0]
        lines.append(
            f"`{art['target']}` is a household expenditure-category proxy for **{commodity_word}**, "
            "not a measurement of physical purchase quantity of that commodity. Treat outputs as "
            "directional/indicative demand pressure, not literal commodity volume."
        )
    lines.append("")
    lines.append("## Verdict")
    lines.append("")
    lines.append(f"**{art['verdict']}** -- {art['verdict_reason']}")
    lines.append("")

    report_path = ARTIFACT_ROOT / "reports" / f"{artifact_name}_validation_report.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print("wrote", report_path)

    demand_validation_rows.append({
        "target": art["target"],
        "artifact_name": artifact_name,
        "selected_model": art["selected_model"],
        "weighted_R2": wm.get("r2"),
        "unweighted_R2": um.get("R2"),
        "WAPE": um.get("WAPE"),
        "sMAPE": um.get("sMAPE"),
        "MASE": um.get("MASE"),
        "weighted_aggregate_error_pct": art["weighted_aggregate_error_pct"],
        "verdict": art["verdict"],
        "verdict_reason": art["verdict_reason"],
    })

demand_validation_summary = pd.DataFrame(demand_validation_rows)
display(demand_validation_summary)
demand_validation_summary.to_csv(ARTIFACT_ROOT / "reports" / "demand_validation_summary.csv", index=False)


wrote artifacts_v3_3\reports\rice_validation_report.md
wrote artifacts_v3_3\reports\vegetable_shared_validation_report.md
wrote artifacts_v3_3\reports\banana_validation_report.md


,target,artifact_name,selected_model,weighted_R2,unweighted_R2,WAPE,sMAPE,MASE,weighted_aggregate_error_pct,verdict,verdict_reason
0,BREAD,rice,XGBRegressor_regularized,0.537046,0.547511,23.236210,23.765643,0.653893,-0.121613,USABLE_PROXY,weighted R2 and/or WAPE meet the usable-proxy bar
1,VEG,vegetable_shared,GradientBoostingRegressor,0.195817,0.194074,43.478839,43.656048,0.862820,-1.472875,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...
2,FRUIT,banana,GradientBoostingRegressor_huber,0.277768,0.294747,47.591572,50.040649,0.778538,-5.470055,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...


## 10. Commodity mapping for the demand proxy

- Rice → `BREAD`
- Banana → `FRUIT`
- Tomato → shared `VEG`
- Red Onion → shared `VEG`

The notebook deliberately **does not create two independent FIES household models for Tomato and Red Onion**, because the FIES summary provides a broad `VEG` category rather than separate tomato/onion expenditure targets.


In [13]:

commodity_demand_registry = {
    "rice":{
        "artifact":"demand/rice.joblib",
        "target":"BREAD",
        "label":"Cereal Household Demand Proxy",
        "commodity_specificity":"category_proxy",
    },
    "tomato":{
        "artifact":"demand/vegetable_shared.joblib",
        "target":"VEG",
        "label":"Vegetable Household Demand Proxy",
        "commodity_specificity":"shared_broad_category",
    },
    "red_onion":{
        "artifact":"demand/vegetable_shared.joblib",
        "target":"VEG",
        "label":"Vegetable Household Demand Proxy",
        "commodity_specificity":"shared_broad_category",
    },
    "banana":{
        "artifact":"demand/banana.joblib",
        "target":"FRUIT",
        "label":"Fruit Household Demand Proxy",
        "commodity_specificity":"category_proxy",
    },
}
(ARTIFACT_ROOT/"config"/"commodity_demand_registry.json").write_text(
    json.dumps(commodity_demand_registry,indent=2),encoding="utf-8"
)


732

## 11. Build a monthly/quarterly CALABARZON LFS economic-activity indicator

In [14]:

# LFS is used only as a temporal socioeconomic indicator.
# It is NOT treated as direct food demand.

lfs_files = [
    p for p in lfs_root.rglob("*.csv")
    if "LFS PUF" in p.name or p.name.lower().startswith("lfs20")
]

def read_lfs_indicator(path):
    try:
        header = pd.read_csv(path,nrows=0).columns.tolist()
    except Exception:
        return None

    required = {"PUFSVYYR","PUFSVYMO","PUFNEWEMPSTAT"}
    if not required.issubset(header):
        return None

    use = [c for c in ["PUFREG","PUFSVYYR","PUFSVYMO","PUFPWGTPRV","PUFNEWEMPSTAT","PUFHHSIZE"] if c in header]
    df=pd.read_csv(path,usecols=use,low_memory=False)

    # Region code 4 is used only where PUFREG is available.
    if "PUFREG" not in df:
        return None
    reg=pd.to_numeric(df["PUFREG"],errors="coerce")
    df=df[reg.eq(4)].copy()
    if df.empty:
        return None

    year=int(pd.to_numeric(df["PUFSVYYR"],errors="coerce").dropna().mode().iloc[0])
    month=int(pd.to_numeric(df["PUFSVYMO"],errors="coerce").dropna().mode().iloc[0])

    emp=pd.to_numeric(df["PUFNEWEMPSTAT"],errors="coerce")
    w=pd.to_numeric(df.get("PUFPWGTPRV",1.0),errors="coerce").fillna(1.0)

    # Coding varies by survey vintage; rather than assume employed-code semantics,
    # derive a stable "work participation share" only when exactly two/three dominant codes exist.
    code_counts=emp.value_counts(normalize=True)
    dominant=code_counts.index[:3].tolist()
    # Most LFS releases code employed persons as 1; verify coverage before using.
    employed=(emp==1).astype(float)
    employment_share=weighted_mean(employed,w)

    hhsize=pd.to_numeric(df.get("PUFHHSIZE",np.nan),errors="coerce")
    mean_hhsize=weighted_mean(hhsize,w) if hasattr(hhsize,"__len__") else np.nan

    return {
        "date":pd.Timestamp(year,month,1),
        "employment_share_proxy":employment_share,
        "mean_household_size":mean_hhsize,
        "rows":len(df),
        "source_file":path.name,
    }

lfs_rows=[]
for p in lfs_files:
    r=read_lfs_indicator(p)
    if r:
        lfs_rows.append(r)

lfs_indicator=pd.DataFrame(lfs_rows).drop_duplicates("date").sort_values("date")
if not lfs_indicator.empty:
    # Normalize around 100; used as an index, not a physical demand quantity.
    base=lfs_indicator["employment_share_proxy"].median()
    lfs_indicator["economic_activity_index"]=lfs_indicator["employment_share_proxy"]/base*100
    lfs_indicator["quarter"]=lfs_indicator["date"].dt.to_period("Q").dt.start_time
    lfs_quarterly=(
        lfs_indicator.groupby("quarter",as_index=False)["economic_activity_index"].mean()
        .rename(columns={"quarter":"date"})
    )
else:
    lfs_quarterly=pd.DataFrame(columns=["date","economic_activity_index"])

display(lfs_indicator.tail(20))
lfs_indicator.to_csv(ARTIFACT_ROOT/"prepared"/"lfs_calabarzon_activity_monthly.csv",index=False)
lfs_quarterly.to_csv(ARTIFACT_ROOT/"prepared"/"lfs_calabarzon_activity_quarterly.csv",index=False)


,date,employment_share_proxy,mean_household_size,rows,source_file,economic_activity_index,quarter
9,2022-12-01,0.423907,5.364140,2335,LFS PUF December 2022.CSV,101.029794,2022-10-01
14,2023-01-01,0.422166,5.330715,9169,LFS PUF January 2023.CSV,100.614861,2023-01-01
12,2023-02-01,0.420761,5.627379,2446,LFS PUF February 2023.CSV,100.279903,2023-01-01
26,2023-03-01,0.412850,5.351059,2330,LFS PUF March 2023.CSV,98.394451,2023-01-01
2,2023-04-01,0.424614,5.394344,9189,LFS PUF April 2023.CSV,101.198182,2023-04-01
29,2023-05-01,0.412214,5.282025,2324,LFS PUF May 2023.CSV,98.243013,2023-04-01
23,2023-06-01,0.430464,5.343078,2317,LFS PUF June 2023.CSV,102.592575,2023-04-01
19,2023-07-01,0.403143,5.223363,36008,LFS PUF July 2023.CSV,96.081130,2023-07-01
7,2023-08-01,0.426975,5.053334,2223,LFS PUF August 2023.CSV,101.761044,2023-07-01
40,2023-09-01,0.416153,4.997556,2183,LFS PUF September 2023.CSV,99.181834,2023-07-01


## 12. Derived quarterly **Demand Pressure Index**

This is the recommended temporal demand representation when no observed quarterly commodity-consumption series exists.

V3.3 combines:

- FIES province household-expenditure baseline
- LFS regional economic-activity index
- commodity-specific seasonal price behavior, expressed as an affordability factor

The commodity's supply-season index is still exported as **context for later supply-gap/opportunity analysis**, but it is deliberately **not multiplied into demand pressure**. Local production going to zero must not imply that household demand also goes to zero.

The components are normalized first, so incompatible units are never directly added/subtracted.

**This is a proxy/index and must be labelled accordingly in the UI.**


In [15]:
def seasonal_profile_quarterly(commodity):
    # Supply seasonal profile is context only in V3.3; it is NOT part of demand pressure.
    s=supply_panels[commodity].dropna(subset=["target"]).copy()
    s_prof=s.groupby("season")["target"].median()
    s_prof=s_prof/s_prof.median()

    # Price -> quarterly affordability profile.
    p=price_panels[commodity].dropna(subset=["target"]).copy()
    p["q"]=p["date"].dt.quarter
    p_prof=p.groupby("q")["target"].median()
    p_prof=p_prof/p_prof.median()
    return s_prof,p_prof

# Province baseline normalized to 100 within each FIES target.
baseline=demand_province_baseline.copy()
baseline["base_index"]=baseline.groupby("target")["predicted_weighted_mean"].transform(
    lambda s:s/s.median()*100
)

pressure_rows=[]
for commodity in COMMODITIES:
    target=DEMAND_TARGETS[commodity]
    b=baseline[baseline["target"].eq(target)]
    sprof,pprof=seasonal_profile_quarterly(commodity)

    for _,br in b.iterrows():
        for _,lr in lfs_quarterly.iterrows():
            q=int(pd.Timestamp(lr["date"]).quarter)
            supply_season=float(sprof.get(q,1.0))
            price_season=float(pprof.get(q,1.0))
            affordability=1.0/max(price_season,1e-6)

            # V3.3: demand pressure is independent from current/local supply.
            # Equal-weight geometric combination of economic activity and affordability only.
            temporal_factor=(lr["economic_activity_index"]/100 * affordability) ** (1/2)

            pressure_rows.append({
                "commodity":commodity,
                "province":br["province"],
                "date":lr["date"],
                "fies_base_index":br["base_index"],
                "economic_activity_index":lr["economic_activity_index"],
                "supply_season_index":supply_season*100,
                "supply_season_role":"context_only_not_in_demand_formula",
                "affordability_season_index":affordability*100,
                "demand_pressure_index":br["base_index"]*temporal_factor,
            })

demand_pressure=pd.DataFrame(pressure_rows)
if not demand_pressure.empty:
    demand_pressure["demand_pressure_index"]=demand_pressure.groupby("commodity")[
        "demand_pressure_index"
    ].transform(lambda s:s/s.median()*100)

display(demand_pressure.tail(20))
demand_pressure.to_csv(ARTIFACT_ROOT/"prepared"/"quarterly_demand_pressure_index.csv",index=False)


,commodity,province,date,fies_base_index,economic_activity_index,supply_season_index,supply_season_role,affordability_season_index,demand_pressure_index
360,banana,Quezon,2025-10-01,69.112515,99.458312,30.090587,context_only_not_in_demand_formula,102.734822,70.087892
361,banana,Rizal,2021-01-01,109.352647,93.260101,44.222920,context_only_not_in_demand_formula,99.652778,105.761807
362,banana,Rizal,2021-04-01,109.352647,98.480630,155.777080,context_only_not_in_demand_formula,98.883682,108.261475
363,banana,Rizal,2021-07-01,109.352647,93.349671,157.355043,context_only_not_in_demand_formula,100.349650,106.181913
364,banana,Rizal,2021-10-01,109.352647,98.386324,30.090587,context_only_not_in_demand_formula,102.734822,110.296675
365,banana,Rizal,2022-01-01,109.352647,99.075478,44.222920,context_only_not_in_demand_formula,99.652778,109.009415
366,banana,Rizal,2022-04-01,109.352647,99.110846,155.777080,context_only_not_in_demand_formula,98.883682,108.607327
367,banana,Rizal,2022-07-01,109.352647,102.209909,157.355043,context_only_not_in_demand_formula,100.349650,111.106803
368,banana,Rizal,2022-10-01,109.352647,104.064576,30.090587,context_only_not_in_demand_formula,102.734822,113.434854
369,banana,Rizal,2023-01-01,109.352647,99.763071,44.222920,context_only_not_in_demand_formula,99.652778,109.387029


### Demand-pressure forecast rule

Because the derived index is not observed demand, V3.3 does **not** report fake MAE/R² against it as though it were ground truth.

For an application forecast:
- use the latest FIES province baseline,
- project regional economic activity conservatively,
- apply the commodity's seasonal price/affordability profile,
- keep supply seasonality separate for the downstream supply-gap/opportunity layer,
- expose the output as **Estimated Demand Pressure Index**,
- show methodology confidence rather than “forecast accuracy”.

A simple seasonal projection can be used for the next three quarters. If later you acquire true quarterly consumption/market purchase data, replace this proxy with a trained and validated demand-forecasting target.


In [16]:
def next_quarter_dates(last_date,n=3):
    p=pd.Timestamp(last_date).to_period("Q")
    return [(p+i).start_time for i in range(1,n+1)]

future_pressure_rows=[]
if not demand_pressure.empty:
    last_date=demand_pressure["date"].max()
    future_dates=next_quarter_dates(last_date,3)

    # Economic activity: conservative recent median until a dedicated LFS forecast is justified.
    econ_recent=lfs_quarterly.tail(4)["economic_activity_index"].median() if not lfs_quarterly.empty else 100.0

    for commodity in COMMODITIES:
        target=DEMAND_TARGETS[commodity]
        b=baseline[baseline["target"].eq(target)]
        sprof,pprof=seasonal_profile_quarterly(commodity)

        for dt in future_dates:
            q=dt.quarter
            supply_season=float(sprof.get(q,1.0))
            price_season=float(pprof.get(q,1.0))
            affordability=1.0/max(price_season,1e-6)
            temporal_factor=(econ_recent/100*affordability)**(1/2)

            for _,br in b.iterrows():
                future_pressure_rows.append({
                    "commodity":commodity,
                    "province":br["province"],
                    "date":dt,
                    "estimated_demand_pressure_index":br["base_index"]*temporal_factor,
                    "supply_season_index_context":supply_season*100,
                    "confidence":"moderate_proxy" if commodity in ["rice","banana"] else "low_to_moderate_shared_veg_proxy",
                    "observed_demand":False,
                })

future_demand_pressure=pd.DataFrame(future_pressure_rows)
if not future_demand_pressure.empty:
    future_demand_pressure["estimated_demand_pressure_index"]=future_demand_pressure.groupby("commodity")[
        "estimated_demand_pressure_index"
    ].transform(lambda s:s/s.median()*100)

display(future_demand_pressure)
future_demand_pressure.to_csv(ARTIFACT_ROOT/"prepared"/"future_demand_pressure_3q.csv",index=False)


,commodity,province,date,estimated_demand_pressure_index,supply_season_index_context,confidence,observed_demand
0,rice,Batangas,2026-01-01,92.003467,110.399866,moderate_proxy,False
1,rice,Cavite,2026-01-01,98.022897,110.399866,moderate_proxy,False
2,rice,Laguna,2026-01-01,102.303840,110.399866,moderate_proxy,False
3,rice,Quezon,2026-01-01,99.816906,110.399866,moderate_proxy,False
4,rice,Rizal,2026-01-01,106.388547,110.399866,moderate_proxy,False
5,rice,Batangas,2026-04-01,92.172230,89.600134,moderate_proxy,False
6,rice,Cavite,2026-04-01,98.202701,89.600134,moderate_proxy,False
7,rice,Laguna,2026-04-01,102.491497,89.600134,moderate_proxy,False
8,rice,Quezon,2026-04-01,100.000000,89.600134,moderate_proxy,False
9,rice,Rizal,2026-04-01,106.583696,89.600134,moderate_proxy,False


## 13. Optional PSA commodity-flow context

In [17]:

# Optional. This section is deliberately not required for model training.
# It can enrich market opportunity with inflow/outflow dependence if internet access is available.

FLOW_URL = "https://openstat.psa.gov.ph/PXWeb/api/v1/en/DB/2L/DTR/QT/0032L4DRQD0.px"

def flow_status_note():
    return {
        "status":"optional_external_indicator",
        "use":"market opportunity / inflow-outflow context",
        "warning":"Section II - Vegetable products is broad unless lower-level PSCC items are explicitly queried.",
        "unit_policy":"convert source volume to metric tons only after confirming source unit metadata",
    }

(ARTIFACT_ROOT/"config"/"commodity_flow_methodology.json").write_text(
    json.dumps(flow_status_note(),indent=2),encoding="utf-8"
)
print(flow_status_note())


{'status': 'optional_external_indicator', 'use': 'market opportunity / inflow-outflow context', 'warning': 'Section II - Vegetable products is broad unless lower-level PSCC items are explicitly queried.', 'unit_policy': 'convert source volume to metric tons only after confirming source unit metadata'}


## 14. Opportunity-index inputs — normalized, never mixed raw units

In [18]:

# The notebook exports components, not a falsely learned "opportunity" target.
# Initial weights remain configurable/expert-defined until observed market outcomes exist.

opportunity_config={
    "schema_version":"3.3",
    "components":{
        "demand_pressure_index":{"weight":0.35,"direction":"higher_better"},
        "supply_gap_or_scarcity_index":{"weight":0.25,"direction":"higher_better"},
        "price_opportunity_index":{"weight":0.20,"direction":"higher_better"},
        "market_flow_dependence_index":{"weight":0.10,"direction":"higher_better","optional":True},
        "forecast_confidence_index":{"weight":0.10,"direction":"higher_better"},
    },
    "policy":[
        "Normalize all inputs to a common 0-100 scale before combination.",
        "Do not subtract FIES pesos directly from supply metric tons.",
        "Keep demand-pressure construction independent from the supply component; combine them only after normalization in the opportunity layer.",
        "Weights are transparent configuration, not learned ground truth.",
    ],
}
(ARTIFACT_ROOT/"config"/"opportunity_scoring_config.json").write_text(
    json.dumps(opportunity_config,indent=2),encoding="utf-8"
)


941

## 15. Unified deployment summary

In [19]:

deployment = pd.concat([
    supply_verdicts.assign(component="supply"),
    price_verdicts.assign(component="price"),
], ignore_index=True, sort=False)

# Add demand-proxy deployment status separately because its validation semantics differ.
demand_rows=[]
for c in COMMODITIES:
    target=DEMAND_TARGETS[c]
    report=demand_model_report[demand_model_report["target"].eq(target)].iloc[0]
    specificity=commodity_demand_registry[c]["commodity_specificity"]
    demand_rows.append({
        "commodity":c,
        "component":"demand_proxy",
        "selected_model":report["selected_model"],
        "R2":report["R2"],
        "MAE":report["MAE"],
        "WAPE":report["WAPE"],
        "MASE":report["MASE"],
        "weighted_aggregate_error_pct":report["weighted_aggregate_error_pct"],
        "verdict":report["verdict"],
        "reason":report["verdict_reason"],
        "specificity":specificity,
    })

deployment_all=pd.concat([deployment,pd.DataFrame(demand_rows)],ignore_index=True,sort=False)
display(deployment_all)
deployment_all.to_csv(ARTIFACT_ROOT/"reports"/"unified_deployment_summary.csv",index=False)


,commodity,kind,selected_model,MAE,MSE,RMSE,R2,MAPE,sMAPE,WAPE,...,rolling_worst_sMAPE,rolling_median_WAPE,rolling_median_MASE,rolling_folds,selection_basis,verdict,reason,component,weighted_aggregate_error_pct,specificity
0,rice,supply,gradient_boosting_huber,2657.071047,2.538374e+07,5038.227611,0.883443,45.436608,33.844399,22.184703,...,52.126097,23.899322,1.029933,12.0,recent rolling-origin validation; final holdou...,CAUTION,moderate forecast error and/or rolling variabi...,supply,NaN,NaN
1,tomato,supply,seasonal_naive,102.245500,7.444939e+04,272.854163,0.968795,23.868586,28.999335,13.447941,...,82.562693,12.062414,1.000000,12.0,recent rolling-origin validation; final holdou...,CAUTION,seasonal-naive baseline is the most robust can...,supply,NaN,NaN
2,red_onion,supply,seasonal_naive,1.261071,8.569416e+00,2.927357,0.515448,1244.634380,64.745796,50.477470,...,200.000000,49.375720,1.000000,12.0,recent rolling-origin validation; final holdou...,INSUFFICIENT_DATA,forecast errors remain too high,supply,NaN,NaN
3,banana,supply,random_forest,571.611678,1.264655e+06,1124.568646,0.936465,15.779644,20.025857,12.855987,...,21.894873,10.072645,0.878733,12.0,recent rolling-origin validation; final holdou...,PASS,,supply,NaN,NaN
4,rice,price,random_forest,1.990229,7.362220e+00,2.713341,0.639951,10.922603,11.146308,11.291915,...,18.525786,7.266526,0.294368,24.0,recent rolling-origin validation; final holdou...,PASS,,price,NaN,NaN
5,tomato,price,hist_gradient_boosting,17.611458,5.216539e+02,22.839744,0.039344,45.731863,37.479223,39.791112,...,114.144740,39.931969,0.742080,24.0,recent rolling-origin validation; final holdou...,CAUTION,moderate forecast error and/or rolling variabi...,price,NaN,NaN
6,red_onion,price,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,INSUFFICIENT_DATA,no candidate could be rolling-validated,price,NaN,NaN
7,banana,price,random_forest,3.827389,2.801419e+01,5.292843,-0.085015,15.275523,14.533540,14.929451,...,38.386793,14.286784,0.686672,24.0,recent rolling-origin validation; final holdou...,CAUTION,absolute/percentage error is acceptable but ne...,price,NaN,NaN
8,rice,NaN,XGBRegressor_regularized,6168.236276,NaN,NaN,0.537046,NaN,NaN,23.236210,...,NaN,NaN,NaN,NaN,NaN,USABLE_PROXY,weighted R2 and/or WAPE meet the usable-proxy bar,demand_proxy,-0.121613,category_proxy
9,tomato,NaN,GradientBoostingRegressor,3426.177015,NaN,NaN,0.195817,NaN,NaN,43.478839,...,NaN,NaN,NaN,NaN,NaN,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...,demand_proxy,-1.472875,shared_broad_category


## 16. Data-quality and methodology registry


In [20]:
methodology={
    "version":"3.3",
    "supply":{
        "frequency":"quarterly",
        "target":"PSA volume of production",
        "auxiliary":["lagged area planted/harvested","lagged yield proxy","lagged farmgate price","province one-hot"],
        "leakage_control":"same-period area and target-derived yield_proxy are excluded from predictors",
    },
    "price":{
        "frequency":"monthly",
        "target":"PSA farmgate price",
        "critical_cleaning":"non-positive prices treated as missing",
        "auxiliary":["lagged supply","lagged area","lagged yield proxy","province one-hot"],
    },
    "time_series_selection":{
        "candidate_families":["seasonal_naive","HistGradientBoosting","HistGradientBoosting-poisson","GradientBoosting-huber","RandomForest"],
        "selection":"dense recent expanding-window rolling origin on pre-holdout training data only",
        "final_holdout":"most recent 4 quarters for supply; most recent 12 months for price",
        "stability_summary":"median and 90th-percentile rolling sMAPE; worst fold retained as diagnostic only",
    },
    "demand":{
        "fies_year":"2023 in supplied household-summary file",
        "targets":{"rice":"BREAD","tomato":"VEG","red_onion":"VEG","banana":"FRUIT"},
        "aggregation":"survey-weighted using RFACT",
        "training":"sample_weight=RFACT passed to model.fit(); target and survey weight excluded from predictors",
        "model_race":"GB vs Huber-GB vs log1p-GB vs XGB vs regularized-XGB vs Poisson-HistGB; selected by 3-fold province+target-quantile stratified CV inside the training partition",
        "final_holdout":"stratified 25% household holdout reserved until candidate family is selected",
        "mase_baseline":"province-mean naive (cross-sectional FIES has no time axis)",
        "verdict_bands":"USABLE_PROXY (weighted R2>=0.45 or WAPE<=30%), INDICATIVE_PROXY (weighted R2>=0.20 or WAPE<=55%), INSUFFICIENT_DATA otherwise",
        "temporal_proxy":"FIES baseline + LFS economic activity + seasonal affordability; supply seasonality is context only and is not multiplied into demand pressure",
        "not_claimed":["physical MT demand","observed quarterly commodity consumption"],
    },
    "validation":{
        "supply_price":"rolling-origin model selection on training history + independent chronological final holdout",
        "demand_household_proxy":"stratified CV model selection + untouched household holdout + province leave-one-out + weighted aggregate error + per-target validation_report.md",
    },
}
(ARTIFACT_ROOT/"reports"/"methodology_registry.json").write_text(
    json.dumps(methodology,indent=2),encoding="utf-8"
)
print(json.dumps(methodology,indent=2))


{
  "version": "3.3",
  "supply": {
    "frequency": "quarterly",
    "target": "PSA volume of production",
    "auxiliary": [
      "lagged area planted/harvested",
      "lagged yield proxy",
      "lagged farmgate price",
      "province one-hot"
    ],
    "leakage_control": "same-period area and target-derived yield_proxy are excluded from predictors"
  },
  "price": {
    "frequency": "monthly",
    "target": "PSA farmgate price",
    "critical_cleaning": "non-positive prices treated as missing",
    "auxiliary": [
      "lagged supply",
      "lagged area",
      "lagged yield proxy",
      "province one-hot"
    ]
  },
  "time_series_selection": {
    "candidate_families": [
      "seasonal_naive",
      "HistGradientBoosting",
      "HistGradientBoosting-poisson",
      "GradientBoosting-huber",
      "RandomForest"
    ],
    "selection": "dense recent expanding-window rolling origin on pre-holdout training data only",
    "final_holdout": "most recent 4 quarters for supply; 

## 17. Application integration rules

### Supply/price

Use a numerical forecast only when the artifact verdict is `PASS` or `CAUTION`.

If `INSUFFICIENT_DATA`:
- keep the commodity visible,
- show latest observed value/history,
- display **Forecast unavailable — Insufficient Data**.

V3.3 artifacts may include `geolocation` as a categorical model feature and lagged agronomic fields. The inference service must construct exactly the feature names listed in each artifact's `features` entry before calling `model.predict()`.

### Demand

Recommended UI label:

> **Estimated Demand Pressure Index**

Subtext:

> FIES/LFS-derived household market-demand proxy. Index values are relative and are not metric tons of consumption.

For Tomato and Red Onion, additionally state that the underlying household proxy is the broad FIES **Vegetable** expenditure category.

### Opportunity

Keep demand and supply independently normalized before combining them in the opportunity layer. Never calculate:

`FIES expenditure pesos − supply metric tons`.

Those units are not commensurable.


## 18. Files to send back after Run All

Please send these for final review:

```text
artifacts_v3_3/reports/
├── supply_data_audit.csv
├── price_data_audit.csv
├── supply_model_metrics.csv
├── supply_rolling_validation.csv
├── supply_deployment_verdicts.csv
├── price_model_metrics.csv
├── price_rolling_validation.csv
├── price_deployment_verdicts.csv
├── demand_proxy_model_metrics.csv
├── demand_validation_summary.csv
├── unified_deployment_summary.csv
└── methodology_registry.json

artifacts_v3_3/prepared/
├── demand_proxy_province_baseline.csv
├── quarterly_demand_pressure_index.csv
└── future_demand_pressure_3q.csv
```

For the next validation pass, check especially:
1. whether rice supply remains strong after removing same-period target-derived yield leakage;
2. whether recent rolling p90 improves the stability interpretation for tomato/banana;
3. whether Poisson/Huber candidates help zero-heavy Red Onion supply or volatile Tomato price;
4. whether the cross-validated FIES candidate race improves VEG/FRUIT without degrading the untouched final holdout;
5. whether Red Onion price remains `INSUFFICIENT_DATA` (likely, unless additional observations exist).


## 19. Promote artifacts for the EF_agriwise forecast service

Copies trained bundles into `EF_agriwise/ml/artifacts/{demand,supply,price}/<slug>.joblib` with Title-Case `commodity`, `model_id`, `verdict`, `metrics`, and `limitations`, so `EF_agriwise/ml/forecasting/artifact_registry.py` can load them directly. Skipped gracefully (with a printed message) if `EF_agriwise` isn't found next to this notebook.


In [ ]:

def promote_bundle(component, out_dir, filename_stem, bundle):
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{filename_stem}.joblib"
    joblib.dump(bundle, out_path)
    print("promoted", component, bundle.get("commodity"), "->", out_path)
    return out_path

if EF_ARTIFACTS_ROOT is None:
    print("Skipping artifact promotion: EF_agriwise was not found relative to this notebook.")
elif not PROMOTE_TO_EF_AGRIWISE:
    print(
        "Skipping artifact promotion: PROMOTE_TO_EF_AGRIWISE is False. "
        "Review this run's deployment-summary verdicts (Section 15) and, if they look good, "
        "set PROMOTE_TO_EF_AGRIWISE = True near ARTIFACT_ROOT and re-run this cell."
    )
else:
    for component in ["supply", "price"]:
        out_dir = EF_ARTIFACTS_ROOT / component
        for path in sorted((ARTIFACT_ROOT / component).glob("*.joblib")):
            bundle = joblib.load(path)
            if "commodity" not in bundle or "model_id" not in bundle or "verdict" not in bundle:
                print("skipping incompatible bundle (missing commodity/model_id/verdict):", path)
                continue
            promote_bundle(component, out_dir, slugify_commodity(bundle["commodity"]), bundle)

    demand_out_dir = EF_ARTIFACTS_ROOT / "demand"
    for artifact_name, commodities in DEMAND_ARTIFACT_COMMODITIES.items():
        src = ARTIFACT_ROOT / "demand" / f"{artifact_name}.joblib"
        if not src.exists():
            print("skipping missing demand artifact:", src)
            continue
        base = joblib.load(src)
        for commodity_title in commodities:
            promoted = dict(base)
            promoted["commodity"] = commodity_title
            promoted["metrics"] = base.get("weighted_metrics", {})
            promote_bundle("demand", demand_out_dir, slugify_commodity(commodity_title), promoted)

    print("Artifact promotion complete ->", EF_ARTIFACTS_ROOT)
